In [ ]:
BASE = '/drive/MyDrive/asia-health-risk-geodatabase/'
import pandas as pd
import geopandas as gpd
import json
import plotly.graph_objects as go

master_risk_final = pd.read_csv(BASE + 'master_risk_final_v2.csv')
master_risk_final['rank'] = range(1, len(master_risk_final) + 1)
master_risk_final['country'] = 'Philippines'

name_map = {
    'Metropolitan Manila First District': 'Manila City (1st District)',
    'Metropolitan Manila Second District': 'Quezon City (2nd District)',
    'Metropolitan Manila Third District': 'Caloocan & Others (3rd District)',
    'Metropolitan Manila Fourth District': 'Pasig & Others (4th District)',
    'Cotabato (North Cotabato)': 'North Cotabato',
    'Samar (Western Samar)': 'Western Samar',
    'Davao de Oro (Compostela Valley)': 'Davao de Oro',
    'City of Isabela (not a province)': 'Isabela City',
    'Special Geographic Area': 'BARMM Spec. Area'
}
master_risk_final['display_name'] = master_risk_final['ADM2_EN'].replace(name_map)
df_sorted = master_risk_final.sort_values('overlap_score', ascending=True)
df_sorted['display_name'] = df_sorted['ADM2_EN'].replace(name_map)

print("ready — provinces:", len(master_risk_final))

In [ ]:
master_risk_final['display_name'] = master_risk_final['ADM2_EN'].replace(name_map)
df_sorted = master_risk_final.sort_values('overlap_score', ascending=True)
df_sorted['display_name'] = df_sorted['ADM2_EN'].replace(name_map)

# verify
print(master_risk_final[master_risk_final['ADM2_EN'].str.contains('Metropolitan', na=False)]['display_name'].tolist())

In [ ]:
# map rebuild cell

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
import pandas as pd

mapbox_token = "pk.eyJ1IjoieW5uZ3V5ZW4iLCJhIjoiY21teDR3NzN2MTE0czJ1b2pjcTJ6d3Y2aCJ9.ATgZczFgyaW9dcHWCBMdzg"
# load geojson
with open(BASE + 'philippines_risk_v3.geojson') as f:
    geojson_v2 = json.load(f)

# name shortening
name_map = {
    'Metropolitan Manila First District': 'Manila City (1st District)',
    'Metropolitan Manila Second District': 'Quezon City (2nd District)',
    'Metropolitan Manila Third District': 'Caloocan & Others (3rd District)',
    'Metropolitan Manila Fourth District': 'Pasig & Others (4th District)',
    'Cotabato (North Cotabato)': 'North Cotabato',
    'Samar (Western Samar)': 'Western Samar',
    'Davao de Oro (Compostela Valley)': 'Davao de Oro',
    'City of Isabela (not a province)': 'Isabela City',
    'Special Geographic Area': 'BARMM Spec. Area'
}

master_risk_final['display_name'] = master_risk_final['ADM2_EN'].replace(name_map)
df_sorted = master_risk_final.sort_values('overlap_score', ascending=True)
df_sorted['display_name'] = df_sorted['ADM2_EN'].replace(name_map)

# build satellite choropleth map
fig_map = go.Figure(go.Choroplethmapbox(
    geojson=geojson_v2,
    locations=master_risk_final['ADM2_PCODE'],
    z=master_risk_final['overlap_score'],
    featureidkey='properties.ADM2_PCODE',
    colorscale='RdYlGn_r',
    zmin=master_risk_final['overlap_score'].min(),
    zmax=master_risk_final['overlap_score'].max(),
    marker_opacity=0.7,
    marker_line_width=0.5,
    marker_line_color='white',
    text=master_risk_final['display_name'],
    customdata=master_risk_final[['pm25','elderly','children_u5','rural_pop_perc','hospital_access_count','rank']].values,
    hovertemplate=(
        '<span style="font-size:15px"><b>%{text}</b></span><br>'
        '<span style="color:#888">Rank %{customdata[5]:.0f} of 88 · Philippines</span><br>'
        '<br>'
        '<b>Risk overlap score: %{z}</b><br>'
        '<br>'
        'PM2.5 (satellite 2022): %{customdata[0]} ug/m3<br>'
        'Elderly population (60+): %{customdata[1]:,}<br>'
        'Children under 5: %{customdata[2]:,}<br>'
        'Rural population: %{customdata[3]}%<br>'
        'Est. pop within 30min of hospital: %{customdata[4]:,}'
        '<extra></extra>'
    ),
    colorbar=dict(
        title=dict(
            text='Health risk<br>overlap score',
            font=dict(size=12),
            side='top'
        ),
        thickness=20,
        len=0.75,
        x=0.98,
        tickfont=dict(size=12),
        tickvals=[
            master_risk_final['overlap_score'].min(),
            master_risk_final['overlap_score'].quantile(0.25),
            master_risk_final['overlap_score'].median(),
            master_risk_final['overlap_score'].quantile(0.75),
            master_risk_final['overlap_score'].max()
        ],
        ticktext=[
            str(round(master_risk_final['overlap_score'].min(), 2)),
            str(round(master_risk_final['overlap_score'].quantile(0.25), 2)),
            str(round(master_risk_final['overlap_score'].median(), 2)),
            str(round(master_risk_final['overlap_score'].quantile(0.75), 2)),
            str(round(master_risk_final['overlap_score'].max(), 2))
        ],
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(0,0,0,0.2)',
        borderwidth=1
    )
))

fig_map.update_layout(
    mapbox=dict(
        accesstoken=mapbox_token,
        style='mapbox://styles/mapbox/satellite-streets-v12',
        center=dict(lat=15, lon=118),
        zoom=4
    ),
    margin=dict(l=0, r=0, t=80, b=0),
    height=650,
    title=dict(
        text='Asia Climate-Health Risk Index<br><sup>Air Pollution · Vulnerability · Healthcare Access Gap · Currently showing: Philippines (88 provinces)</sup>',
        x=0.5,
        xanchor='center',
        font=dict(size=16)
    ),
    hoverlabel=dict(
        bgcolor='white',
        bordercolor='#1B5E8A',
        font=dict(size=13, family='Arial'),
        align='left'
    )
)

# build bar chart
fig_bar = go.Figure(go.Bar(
    x=df_sorted['overlap_score'],
    y=df_sorted['display_name'],
    orientation='h',
    marker=dict(
        color=df_sorted['overlap_score'],
        colorscale='RdYlGn_r',
        showscale=False
    ),
    text=df_sorted['overlap_score'],
    textposition='outside',
    customdata=df_sorted[['pm25','elderly','children_u5','hospital_access_count']].values,
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Risk overlap score: %{x}<br>'
        'PM2.5: %{customdata[0]} ug/m3<br>'
        'Elderly (60+): %{customdata[1]:,}<br>'
        'Children under 5: %{customdata[2]:,}<br>'
        'Est. pop within 30min of hospital: %{customdata[3]:,}'
        '<extra></extra>'
    )
))

fig_bar.update_layout(
    height=1800,
    margin=dict(l=180, r=100, t=80, b=40),
    title=dict(
        text='Provincial Risk Ranking — All 88 Provinces<br><sup>Higher score = greater compounded health risk</sup>',
        x=0.5,
        xanchor='center',
        font=dict(size=16)
    ),
    xaxis=dict(title='Risk overlap score', range=[0, 0.85]),
    yaxis=dict(tickfont=dict(size=11))
)

fig_map.show()
print("saved successfully")


In [ ]:
# pandas is a library that handles tables of data — like Excel but in code
import pandas as pd

# read_csv loads your file into memory
df = pd.read_csv('/content/climate_air_quality.csv')

# extract first 6 characters of the P-code = province level
df['adm2_pco'] = df['adm4_pco'].str[:6]

# group all barangay rows by province, then average the pollution values
adm2 = df.groupby('adm2_pco')[['pm25', 'no2', 'so2']].mean().reset_index()

# round to 2 decimal places so numbers are clean
adm2 = adm2.round(2)

# preview the first 10 rows so you can check it looks right
print(adm2.head(10))

# save the result as a new clean CSV
adm2.to_csv('air_quality_adm2.csv', index=False)

In [ ]:
print(df.columns.tolist())

In [ ]:
import pandas as pd

df = pd.read_csv('/content/climate_air_quality.csv')

# extract first 6 characters of the P-code = province level
df['adm2_pco'] = df['adm4_pcode'].str[:6]

# group all barangay rows by province, then average the pollution values
adm2 = df.groupby('adm2_pco')[['pm25', 'no2', 'so2']].mean().reset_index()

# round to 2 decimal places so numbers are clean
adm2 = adm2.round(2)

# preview the first 10 rows
print(adm2.head(10))

# save the clean output
adm2.to_csv('air_quality_adm2.csv', index=False)

In [ ]:
vuln = pd.read_csv('/content/PHL_ADM2_vulnerability.csv')
print(vuln.columns.tolist())
print(vuln.shape)

In [ ]:
vuln = pd.read_csv('/content/PHL_ADM2_vulnerability.csv')
demo = pd.read_csv('/content/PHL_ADM2_demographics.csv')
acc  = pd.read_csv('/content/PHL_ADM2_access.csv')

print("VULNERABILITY:", vuln.columns.tolist(), vuln.shape)
print("DEMOGRAPHICS:", demo.columns.tolist(), demo.shape)
print("ACCESS:", acc.columns.tolist(), acc.shape)

In [ ]:
df['adm2_pco'] = df['adm4_pcode'].str[:6]
adm2 = df.groupby('adm2_pco')[['pm25', 'no2', 'co']].mean().reset_index()
adm2 = adm2.round(2)
print(adm2.head(5))

In [ ]:
# keep only the columns we need
vuln_clean = vuln[['ADM2_PCODE', 'elderly', 'rural_pop_perc']]

acc_clean = acc[['ADM2_PCODE', 'access_pop_hospitals_30min']].copy()
acc_clean['health_service_gap'] = 100 - acc_clean['access_pop_hospitals_30min']
acc_clean = acc_clean[['ADM2_PCODE', 'health_service_gap']]

air_clean = adm2[['adm2_pco', 'pm25', 'no2', 'co']].rename(columns={'adm2_pco': 'ADM2_PCODE'})

# merge all three into one master table
master = vuln_clean.merge(acc_clean, on='ADM2_PCODE').merge(air_clean, on='ADM2_PCODE')

# preview
print(master.shape)
print(master.head(5))

In [ ]:
print("vulnerability sample:", vuln_clean['ADM2_PCODE'].head(3).tolist())
print("access sample:", acc_clean['ADM2_PCODE'].head(3).tolist())
print("air quality sample:", air_clean['ADM2_PCODE'].head(3).tolist())

In [ ]:
# re-aggregate air quality with 7 character P-code to match other files
df['adm2_pco'] = df['adm4_pcode'].str[:7]
adm2 = df.groupby('adm2_pco')[['pm25', 'no2', 'co']].mean().reset_index()
adm2 = adm2.round(2)

# rename to match
air_clean = adm2[['adm2_pco', 'pm25', 'no2', 'co']].rename(columns={'adm2_pco': 'ADM2_PCODE'})

# now retry the merge
master = vuln_clean.merge(acc_clean, on='ADM2_PCODE').merge(air_clean, on='ADM2_PCODE')

print(master.shape)
print(master.head(5))

In [ ]:
print("air quality sample:", air_clean['ADM2_PCODE'].head(10).tolist())
print("vulnerability sample:", vuln_clean['ADM2_PCODE'].head(10).tolist())
print("total air quality provinces:", len(air_clean))
print("total vulnerability provinces:", len(vuln_clean))

In [ ]:
# look at 10 raw barangay P-codes from the original file
print(df['adm4_pcode'].head(10).tolist())

# also check how long they are
print("P-code length:", df['adm4_pcode'].str.len().value_counts())

In [ ]:
print(df.columns.tolist())
print(vuln.columns.tolist())

In [ ]:
import requests
import io

# official OCHA Philippines admin2 crosswalk
url = "https://data.humdata.org/dataset/cod-ab-phl/resource/12457589-6426-4f84-a858-e3b3fc8a0f1f/download/phl_adminboundaries_tabulardata.xlsx"

response = requests.get(url)
crosswalk = pd.read_excel(io.BytesIO(response.content), sheet_name='ADM2')

print(crosswalk.columns.tolist())
print(crosswalk.head(3))

In [ ]:
crosswalk = pd.read_excel(io.BytesIO(response.content), sheet_name='ADM2', engine='openpyxl')
print(crosswalk.columns.tolist())
print(crosswalk.head(3))

In [ ]:
crosswalk_raw = pd.read_excel('/content/phl_adminboundaries_tabulardata.xlsx',
                               sheet_name='ADM2',
                               engine='openpyxl')
print(crosswalk_raw.columns.tolist())
print(crosswalk_raw.head(3))

In [ ]:
# look at just the columns we need
crosswalk = crosswalk_raw[['ADM2_EN', 'ADM2_PCODE']]

# check how many air quality P-codes match anything in the crosswalk
air_pcodes = set(adm2['adm2_pco'].tolist())
cross_pcodes = set(crosswalk['ADM2_PCODE'].tolist())

print("air quality P-codes:", air_pcodes)
print("---")
print("any matches?", air_pcodes.intersection(cross_pcodes))

adm2['region_code'] = adm2['adm2_pco'].str[2:4]
crosswalk['region_code'] = crosswalk['ADM2_PCODE'].str[2:4]

print("air quality regions:", sorted(adm2['region_code'].unique().tolist()))
print("crosswalk regions:", sorted(crosswalk['region_code'].unique().tolist()))

# reverse the last 3 digits of the air quality P-code and see if it matches
adm2['pcode_test'] = adm2['adm2_pco'].str[:4] + adm2['adm2_pco'].str[4:7].str[::-1]

print(adm2[['adm2_pco', 'pcode_test']])
print("---")
print("matches after reversal:", set(adm2['pcode_test']).intersection(set(crosswalk['ADM2_PCODE'])))

In [ ]:
print(crosswalk[crosswalk['region_code'] == '01'][['ADM2_EN', 'ADM2_PCODE']])

In [ ]:
# manual crosswalk — air quality code to official ADM2_PCODE
manual_crosswalk = pd.DataFrame({
    'adm2_pco': ['PH01551','PH03491','PH05050','PH06302','PH07223',
                 'PH08374','PH09733','PH10430','PH11240','PH13740',
                 'PH13750','PH13760'],
    'ADM2_PCODE': ['PH01055','PH03046','PH05050','PH06203','PH07223',
                   'PH08374','PH09733','PH10430','PH11240','PH13047',
                   'PH13057','PH13067']
})

# verify against crosswalk
matches = manual_crosswalk['ADM2_PCODE'].isin(crosswalk['ADM2_PCODE'])
print("matched provinces:", matches.sum(), "out of 12")
print(manual_crosswalk[matches].merge(crosswalk, on='ADM2_PCODE')[['adm2_pco','ADM2_PCODE','ADM2_EN']])

In [ ]:
print(crosswalk[['ADM2_EN', 'ADM2_PCODE']].to_string())

In [ ]:
region_names = {
    '01': 'Region I', '03': 'Region III', '05': 'Region V',
    '06': 'Region VI', '07': 'Region VII', '08': 'Region VIII',
    '09': 'Region IX', '10': 'Region X', '11': 'Region XI', '13': 'NCR'
}

adm2['region'] = adm2['adm2_pco'].str[2:4].map(region_names)
print(adm2[['adm2_pco', 'region', 'pm25', 'no2', 'co']])

In [ ]:
manual_crosswalk = pd.DataFrame({
    'adm2_pco': ['PH01551','PH03491','PH05050','PH06302','PH07223',
                 'PH08374','PH09733','PH10430','PH11240','PH13740',
                 'PH13750','PH13760'],
    'ADM2_PCODE': ['PH01055','PH03049','PH05050','PH06030','PH07022',
                   'PH08037','PH09073','PH10043','PH11024','PH13074',
                   'PH13075','PH13076']
})

# verify matches
matches = manual_crosswalk['ADM2_PCODE'].isin(crosswalk['ADM2_PCODE'])
print("matched:", matches.sum(), "out of 12")
verified = manual_crosswalk[matches].merge(crosswalk, on='ADM2_PCODE')[['adm2_pco','ADM2_PCODE','ADM2_EN']]
print(verified)

In [ ]:
# bridge air quality to standard P-codes
adm2_bridged = adm2.merge(manual_crosswalk, on='adm2_pco')
air_clean = adm2_bridged[['ADM2_PCODE', 'pm25', 'no2', 'co']]

# final master merge
master = vuln_clean.merge(acc_clean, on='ADM2_PCODE').merge(air_clean, on='ADM2_PCODE')

print(master.shape)
print(master.head(5))

In [ ]:
print(acc[['ADM2_PCODE', 'access_pop_hospitals_30min']].head(10))
print("\nmin:", acc['access_pop_hospitals_30min'].min())
print("max:", acc['access_pop_hospitals_30min'].max())

In [ ]:
print(demo.columns.tolist())
print(demo.head(3))
```

We need a total population column from `demo`. It's likely called something like `total_pop` or we can derive it from the columns we already saw: `female_pop`, `pop_u15` etc. Once we see what's there I'll write the corrected formula — it'll be something like:
```
health_service_gap = (total_pop - access_pop_hospitals_30min) / total_pop × 100

In [ ]:
print(demo.columns.tolist())
print(demo.head(3))

In [ ]:
# estimate total population from female population
demo['total_pop'] = demo['female_pop'] * 2

# rebuild access with corrected gap percentage
acc_clean = acc[['ADM2_PCODE', 'access_pop_hospitals_30min']].merge(
    demo[['ADM2_PCODE', 'total_pop']], on='ADM2_PCODE'
)
acc_clean['health_service_gap'] = ((acc_clean['total_pop'] - acc_clean['access_pop_hospitals_30min']) / acc_clean['total_pop'] * 100).round(2)
acc_clean = acc_clean[['ADM2_PCODE', 'health_service_gap']]

# rebuild vulnerability
vuln_clean = vuln[['ADM2_PCODE', 'elderly', 'rural_pop_perc']]

# rebuild air clean
adm2_bridged = adm2.merge(manual_crosswalk, on='adm2_pco')
air_clean = adm2_bridged[['ADM2_PCODE', 'pm25', 'no2', 'co']]

# final merge
master = vuln_clean.merge(acc_clean, on='ADM2_PCODE').merge(air_clean, on='ADM2_PCODE')

print(master.shape)
print(master.head(5))

In [ ]:
master.to_csv('master_risk_data.csv', index=False)
print("saved successfully")

In [ ]:
# use access count directly, normalised — higher value means BETTER access
# we'll invert it during scoring so high access = low risk
acc_clean = acc[['ADM2_PCODE', 'access_pop_hospitals_30min']].copy()
acc_clean = acc_clean.rename(columns={'access_pop_hospitals_30min': 'hospital_access_count'})

# rebuild everything
vuln_clean = vuln[['ADM2_PCODE', 'elderly', 'rural_pop_perc']]
adm2_bridged = adm2.merge(manual_crosswalk, on='adm2_pco')
air_clean = adm2_bridged[['ADM2_PCODE', 'pm25', 'no2', 'co']]

# final merge
master = vuln_clean.merge(acc_clean, on='ADM2_PCODE').merge(air_clean, on='ADM2_PCODE')

# save to CSV
master.to_csv('master_risk_data.csv', index=False)

print(master.shape)
print(master)

In [ ]:
# add province names from crosswalk
master_named = master.merge(crosswalk[['ADM2_PCODE', 'ADM2_EN']], on='ADM2_PCODE')

# reorder columns so name appears first
master_named = master_named[['ADM2_PCODE', 'ADM2_EN', 'elderly', 'rural_pop_perc',
                              'hospital_access_count', 'pm25', 'no2', 'co']]

print(master_named)

# save
master_named.to_csv('master_risk_data.csv', index=False)
print("\nsaved successfully")

In [ ]:
# normalise each indicator to 0-1 scale
master_named['pm25_norm'] = (master_named['pm25'] - master_named['pm25'].min()) / (master_named['pm25'].max() - master_named['pm25'].min())

master_named['elderly_norm'] = (master_named['elderly'] - master_named['elderly'].min()) / (master_named['elderly'].max() - master_named['elderly'].min())

# for hospital access, MORE access = LOWER risk so we invert it
master_named['access_norm'] = 1 - (master_named['hospital_access_count'] - master_named['hospital_access_count'].min()) / (master_named['hospital_access_count'].max() - master_named['hospital_access_count'].min())

# weighted overlap score
master_named['overlap_score'] = (
    master_named['pm25_norm'] * 0.40 +
    master_named['elderly_norm'] * 0.30 +
    master_named['access_norm'] * 0.30
).round(3)

# sort by highest risk first
master_named = master_named.sort_values('overlap_score', ascending=False)

print(master_named[['ADM2_EN', 'pm25', 'elderly', 'hospital_access_count', 'overlap_score']])

# save final version
master_named.to_csv('master_risk_final.csv', index=False)
print("\nsaved successfully")

In [ ]:
import plotly.express as px
import json

# load the geojson file
with open('/content/adm2_simplified.geojson') as f:
    geojson = json.load(f)

# create the choropleth map
fig = px.choropleth(
    master_risk_final,
    geojson=geojson,
    locations='ADM2_PCODE',
    featureidkey='properties.ADM2_PCODE',
    color='overlap_score',
    hover_name='ADM2_EN',
    hover_data=['pm25', 'elderly', 'hospital_access_count'],
    color_continuous_scale='RdYlGn_r',
    title='Philippines Health Risk Overlap — Air Pollution, Vulnerability and Healthcare Access'
)

fig.update_geos(fitbounds='locations', visible=False)
fig.show()

import pandas as pd

master_risk_final = pd.read_csv('/content/master_risk_final.csv')
print(master_risk_final.shape)
print(master_risk_final.columns.tolist())

In [ ]:
import pandas as pd

master_risk_final = pd.read_csv('/content/master_risk_final.csv')
print(master_risk_final.shape)
print(master_risk_final.columns.tolist())

In [ ]:
import plotly.express as px
import json

# load the geojson file
with open('/content/adm2_simplified.geojson') as f:
    geojson = json.load(f)

# create the choropleth map
fig = px.choropleth(
    master_risk_final,
    geojson=geojson,
    locations='ADM2_PCODE',
    featureidkey='properties.ADM2_PCODE',
    color='overlap_score',
    hover_name='ADM2_EN',
    hover_data=['pm25', 'elderly', 'hospital_access_count'],
    color_continuous_scale='RdYlGn_r',
    title='Philippines Health Risk Overlap — Air Pollution, Vulnerability and Healthcare Access'
)

fig.update_geos(fitbounds='locations', visible=False)
fig.show()

fig.write_html('philippines_health_risk_map.html')
print("saved successfully")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# sort by overlap score highest to lowest
df_sorted = master_risk_final.sort_values('overlap_score', ascending=True)

# create figure with two panels side by side
fig2 = make_subplots(
    rows=1, cols=2,
    column_widths=[0.6, 0.4],
    specs=[[{'type': 'choropleth'}, {'type': 'bar'}]],
    subplot_titles=['Risk overlap by province', 'Ranked overlap score']
)

# add choropleth map on the left
fig2.add_trace(
    go.Choropleth(
        geojson=geojson,
        locations=master_risk_final['ADM2_PCODE'],
        z=master_risk_final['overlap_score'],
        featureidkey='properties.ADM2_PCODE',
        colorscale='RdYlGn_r',
        showscale=True,
        text=master_risk_final['ADM2_EN'],
        hovertemplate='<b>%{text}</b><br>Overlap score: %{z}<extra></extra>'
    ),
    row=1, col=1
)

# add bar chart on the right
fig2.add_trace(
    go.Bar(
        x=df_sorted['overlap_score'],
        y=df_sorted['ADM2_EN'],
        orientation='h',
        marker_color=df_sorted['overlap_score'],
        marker_colorscale='RdYlGn_r',
        text=df_sorted['overlap_score'],
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>Score: %{x}<extra></extra>'
    ),
    row=1, col=2
)

fig2.update_geos(fitbounds='locations', visible=False)
fig2.update_layout(
    title='Philippines Health Risk Overlap Dashboard',
    height=600,
    showlegend=False
)

fig2.write_html('philippines_health_risk_dashboard.html')
fig2.show()
print("saved successfully")

In [ ]:
import pandas as pd
import json

master_risk_final = pd.read_csv('/content/master_risk_final.csv')

with open('/content/adm2_simplified.geojson') as f:
    geojson = json.load(f)

print("reloaded successfully")
print(master_risk_final.shape)

In [ ]:
import pandas as pd
import json

master_risk_final = pd.read_csv('/content/master_risk_final.csv')

with open('/content/adm2_simplified.geojson') as f:
    geojson = json.load(f)

print("reloaded successfully")
print(master_risk_final.shape)

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# sort by overlap score highest to lowest
df_sorted = master_risk_final.sort_values('overlap_score', ascending=True)

# create figure with two panels side by side
fig2 = make_subplots(
    rows=1, cols=2,
    column_widths=[0.6, 0.4],
    specs=[[{'type': 'choropleth'}, {'type': 'bar'}]],
    subplot_titles=['Risk overlap by province', 'Ranked overlap score']
)

# add choropleth map on the left
fig2.add_trace(
    go.Choropleth(
        geojson=geojson,
        locations=master_risk_final['ADM2_PCODE'],
        z=master_risk_final['overlap_score'],
        featureidkey='properties.ADM2_PCODE',
        colorscale='RdYlGn_r',
        showscale=True,
        text=master_risk_final['ADM2_EN'],
        hovertemplate='<b>%{text}</b><br>Overlap score: %{z}<extra></extra>'
    ),
    row=1, col=1
)

# add bar chart on the right
fig2.add_trace(
    go.Bar(
        x=df_sorted['overlap_score'],
        y=df_sorted['ADM2_EN'],
        orientation='h',
        marker_color=df_sorted['overlap_score'],
        marker_colorscale='RdYlGn_r',
        text=df_sorted['overlap_score'],
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>Score: %{x}<extra></extra>'
    ),
    row=1, col=2
)

fig2.update_geos(fitbounds='locations', visible=False)
fig2.update_layout(
    title='Philippines Health Risk Overlap Dashboard',
    height=600,
    showlegend=False
)

fig2.write_html('philippines_health_risk_dashboard.html')
fig2.show()
print("saved successfully")

In [ ]:
fig2 = make_subplots(
    rows=1, cols=2,
    column_widths=[0.6, 0.4],
    specs=[[{'type': 'choropleth'}, {'type': 'bar'}]],
    subplot_titles=['Risk overlap by province', 'Ranked overlap score']
)

fig2.add_trace(
    go.Choropleth(
        geojson=geojson,
        locations=master_risk_final['ADM2_PCODE'],
        z=master_risk_final['overlap_score'],
        featureidkey='properties.ADM2_PCODE',
        colorscale='RdYlGn_r',
        showscale=True,
        text=master_risk_final['ADM2_EN'],
        customdata=master_risk_final[['pm25','no2','co','elderly','rural_pop_perc','hospital_access_count']].values,
        hovertemplate=(
            '<b>%{text}</b><br>'
            'Overlap score: %{z}<br>'
            'PM2.5: %{customdata[0]} μg/m³<br>'
            'NO2: %{customdata[1]} μg/m³<br>'
            'CO: %{customdata[2]} μg/m³<br>'
            'Elderly population: %{customdata[3]:,}<br>'
            'Rural population: %{customdata[4]}%<br>'
            'Hospital access count: %{customdata[5]:,}'
            '<extra></extra>'
        )
    ),
    row=1, col=1
)

fig2.add_trace(
    go.Bar(
        x=df_sorted['overlap_score'],
        y=df_sorted['ADM2_EN'],
        orientation='h',
        marker_color=df_sorted['overlap_score'],
        marker_colorscale='RdYlGn_r',
        text=df_sorted['overlap_score'],
        textposition='outside',
        customdata=df_sorted[['pm25','no2','co','elderly','hospital_access_count']].values,
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Overlap score: %{x}<br>'
            'PM2.5: %{customdata[0]} μg/m³<br>'
            'NO2: %{customdata[1]} μg/m³<br>'
            'CO: %{customdata[2]} μg/m³<br>'
            'Elderly: %{customdata[3]:,}<br>'
            'Hospital access: %{customdata[4]:,}'
            '<extra></extra>'
        )
    ),
    row=1, col=2
)

fig2.update_geos(fitbounds='locations', visible=False)
fig2.update_layout(
    title='Philippines Health Risk Overlap Dashboard',
    height=600,
    showlegend=False
)

fig2.write_html('philippines_health_risk_dashboard.html')
fig2.show()
print("saved successfully")

In [ ]:
fig2 = make_subplots(
    rows=1, cols=2,
    column_widths=[0.6, 0.4],
    specs=[[{'type': 'choropleth'}, {'type': 'bar'}]],
    subplot_titles=['Risk overlap by province', 'Ranked overlap score']
)

fig2.add_trace(
    go.Choropleth(
        geojson=geojson,
        locations=master_risk_final['ADM2_PCODE'],
        z=master_risk_final['overlap_score'],
        featureidkey='properties.ADM2_PCODE',
        colorscale='RdYlGn_r',
        showscale=True,
        text=master_risk_final['ADM2_EN'],
        customdata=master_risk_final[['pm25','no2','co','elderly','rural_pop_perc','hospital_access_count']].values,
        hovertemplate=(
            '<b>%{text}</b><br>'
            'Overlap score: %{z}<br>'
            'PM2.5: %{customdata[0]} μg/m³<br>'
            'NO2: %{customdata[1]} μg/m³<br>'
            'CO: %{customdata[2]} μg/m³<br>'
            'Elderly population: %{customdata[3]:,}<br>'
            'Rural population: %{customdata[4]}%<br>'
            'Est. population within 30min of hosp

In [ ]:
fig2 = make_subplots(
    rows=1, cols=2,
    column_widths=[0.6, 0.4],
    specs=[[{'type': 'choropleth'}, {'type': 'bar'}]],
    subplot_titles=['Risk overlap by province', 'Ranked overlap score']
)

fig2.add_trace(
    go.Choropleth(
        geojson=geojson,
        locations=master_risk_final['ADM2_PCODE'],
        z=master_risk_final['overlap_score'],
        featureidkey='properties.ADM2_PCODE',
        colorscale='RdYlGn_r',
        showscale=True,
        text=master_risk_final['ADM2_EN'],
        customdata=master_risk_final[['pm25','no2','co','elderly','rural_pop_perc','hospital_access_count']].values,
        hovertemplate='<b>%{text}</b><br>Overlap score: %{z}<br>PM2.5: %{customdata[0]} ug/m3<br>NO2: %{customdata[1]} ug/m3<br>CO: %{customdata[2]} ug/m3<br>Elderly population: %{customdata[3]:,}<br>Rural population: %{customdata[4]}%<br>Est. pop within 30min of hospital: %{customdata[5]:,}<extra></extra>'
    ),
    row=1, col=1
)

fig2.add_trace(
    go.Bar(
        x=df_sorted['overlap_score'],
        y=df_sorted['ADM2_EN'],
        orientation='h',
        marker_color=df_sorted['overlap_score'],
        marker_colorscale='RdYlGn_r',
        text=df_sorted['overlap_score'],
        textposition='outside',
        customdata=df_sorted[['pm25','no2','co','elderly','hospital_access_count']].values,
        hovertemplate='<b>%{y}</b><br>Overlap score: %{x}<br>PM2.5: %{customdata[0]} ug/m3<br>NO2: %{customdata[1]} ug/m3<br>CO: %{customdata[2]} ug/m3<br>Elderly population: %{customdata[3]:,}<br>Est. pop within 30min of hospital: %{customdata[4]:,}<extra></extra>'
    ),
    row=1, col=2
)

fig2.update_geos(fitbounds='locations', visible=False)
fig2.update_layout(
    title='Philippines Health Risk Overlap Dashboard',
    height=600,
    showlegend=False
)

fig2.write_html('philippines_health_risk_dashboard.html')
fig2.show()
print("saved successfully")

In [ ]:
# shorten Metro Manila names for chart readability
df_sorted['display_name'] = master_risk_final['ADM2_EN'].replace({
    'Metropolitan Manila Second District': 'MM 2nd District',
    'Metropolitan Manila Third District': 'MM 3rd District',
    'Metropolitan Manila Fourth District': 'MM 4th District'
})
df_sorted = df_sorted.sort_values('overlap_score', ascending=True)

master_risk_final['display_name'] = master_risk_final['ADM2_EN'].replace({
    'Metropolitan Manila Second District': 'MM 2nd District',
    'Metropolitan Manila Third District': 'MM 3rd District',
    'Metropolitan Manila Fourth District': 'MM 4th District'
})

fig2 = make_subplots(
    rows=1, cols=2,
    column_widths=[0.6, 0.4],
    specs=[[{'type': 'choropleth'}, {'type': 'bar'}]],
    subplot_titles=[
        'Health risk overlap by province',
        'Ranked health risk overlap score'
    ]
)

fig2.add_trace(
    go.Choropleth(
        geojson=geojson,
        locations=master_risk_final['ADM2_PCODE'],
        z=master_risk_final['overlap_score'],
        featureidkey='properties.ADM2_PCODE',
        colorscale='RdYlGn_r',
        showscale=True,
        text=master_risk_final['display_name'],
        customdata=master_risk_final[['pm25','no2','co','elderly','rural_pop_perc','hospital_access_count']].values,
        hovertemplate='<b>%{text}</b><br>Health risk overlap score: %{z}<br>PM2.5 (fine particles): %{customdata[0]} ug/m3<br>NO2 (nitrogen dioxide): %{customdata[1]} ug/m3<br>CO (carbon monoxide): %{customdata[2]} ug/m3<br>Elderly population (60+): %{customdata[3]:,}<br>Rural population: %{customdata[4]}%<br>Est. pop within 30min of hospital: %{customdata[5]:,}<extra></extra>'
    ),
    row=1, col=1
)

fig2.add_trace(
    go.Bar(
        x=df_sorted['overlap_score'],
        y=df_sorted['display_name'],
        orientation='h',
        marker_color=df_sorted['overlap_score'],
        marker_colorscale='RdYlGn_r',
        text=df_sorted['overlap_score'],
        textposition='outside',
        customdata=df_sorted[['pm25','no2','co','elderly','hospital_access_count']].values,
        hovertemplate='<b>%{y}</b><br>Health risk overlap score: %{x}<br>PM2.5 (fine particles): %{customdata[0]} ug/m3<br>NO2 (nitrogen dioxide): %{customdata[1]} ug/m3<br>CO (carbon monoxide): %{customdata[2]} ug/m3<br>Elderly population (60+): %{customdata[3]:,}<br>Est. pop within 30min of hospital: %{customdata[4]:,}<extra></extra>'
    ),
    row=1, col=2
)

fig2.update_geos(fitbounds='locations', visible=False)
fig2.update_layout(
    title=dict(
        text='Philippines Climate-Health Risk Index — Air Pollution, Vulnerability & Healthcare Access Gap<br><sup>Data coverage: 11 of 82 provinces (ground sensor monitoring stations only)</sup>',
        x=0.5,
        xanchor='center'
    ),
    height=600,
    showlegend=False
)

fig2.write_html('philippines_health_risk_dashboard.html')
fig2.show()
print("saved successfully")

In [ ]:
print(vuln.columns.tolist())
print(vuln[['ADM2_PCODE', 'children_u5']].head(5))

In [ ]:
import pandas as pd
import json

# reload all dataframes
master_risk_final = pd.read_csv('/content/master_risk_final.csv')
vuln = pd.read_csv('/content/PHL_ADM2_vulnerability.csv')
acc = pd.read_csv('/content/PHL_ADM2_access.csv')
df = pd.read_csv('/content/climate_air_quality.csv')
crosswalk_raw = pd.read_excel('/content/phl_adminboundaries_tabulardata.xlsx',
                               sheet_name='ADM2', engine='openpyxl')
crosswalk = crosswalk_raw[['ADM2_EN', 'ADM2_PCODE']]

# rebuild air quality aggregation
df['adm2_pco'] = df['adm4_pcode'].str[:7]
adm2 = df.groupby('adm2_pco')[['pm25', 'no2', 'co']].mean().reset_index()
adm2 = adm2.round(2)

# rebuild manual crosswalk
manual_crosswalk = pd.DataFrame({
    'adm2_pco': ['PH01551','PH03491','PH06302','PH07223',
                 'PH08374','PH09733','PH10430','PH11240',
                 'PH13740','PH13750','PH13760'],
    'ADM2_PCODE': ['PH01055','PH03049','PH06030','PH07022',
                   'PH08037','PH09073','PH10043','PH11024',
                   'PH13074','PH13075','PH13076']
})

# reload geojson
with open('/content/adm2_simplified.geojson') as f:
    geojson = json.load(f)

print("all variables reloaded successfully")
print("vuln columns:", vuln.columns.tolist())

In [ ]:
from google.colab import drive
drive.mount('/drive')

In [ ]:
import os
files = os.listdir('/drive/MyDrive/philippines_risk_project')
print(files)

In [ ]:
import pandas as pd
import json

master_risk_final = pd.read_csv('/drive/MyDrive/philippines_risk_project/master_risk_final.csv')
vuln = pd.read_csv('/drive/MyDrive/philippines_risk_project/PHL_ADM2_vulnerability.csv')
acc = pd.read_csv('/drive/MyDrive/philippines_risk_project/PHL_ADM2_access.csv')
df = pd.read_csv('/drive/MyDrive/philippines_risk_project/climate_air_quality.csv')
crosswalk_raw = pd.read_excel('/drive/MyDrive/philippines_risk_project/phl_adminboundaries_tabulardata.xlsx',
                               sheet_name='ADM2', engine='openpyxl')
crosswalk = crosswalk_raw[['ADM2_EN', 'ADM2_PCODE']]

with open('/drive/MyDrive/philippines_risk_project/adm2_simplified.geojson') as f:
    geojson = json.load(f)

# rebuild air quality aggregation
df['adm2_pco'] = df['adm4_pcode'].str[:7]
adm2 = df.groupby('adm2_pco')[['pm25', 'no2', 'co']].mean().reset_index()
adm2 = adm2.round(2)

# rebuild manual crosswalk
manual_crosswalk = pd.DataFrame({
    'adm2_pco': ['PH01551','PH03491','PH06302','PH07223',
                 'PH08374','PH09733','PH10430','PH11240',
                 'PH13740','PH13750','PH13760'],
    'ADM2_PCODE': ['PH01055','PH03049','PH06030','PH07022',
                   'PH08037','PH09073','PH10043','PH11024',
                   'PH13074','PH13075','PH13076']
})

print("all variables reloaded successfully")
print("vuln columns:", vuln.columns.tolist())

In [ ]:
# rebuild master with children_u5 added
vuln_clean = vuln[['ADM2_PCODE', 'elderly', 'rural_pop_perc', 'children_u5']]

acc_clean = acc[['ADM2_PCODE', 'access_pop_hospitals_30min']].copy()
acc_clean['hospital_access_count'] = acc_clean['access_pop_hospitals_30min']
acc_clean = acc_clean[['ADM2_PCODE', 'hospital_access_count']]

adm2_bridged = adm2.merge(manual_crosswalk, on='adm2_pco')
air_clean = adm2_bridged[['ADM2_PCODE', 'pm25', 'no2', 'co']]

master_named = vuln_clean.merge(acc_clean, on='ADM2_PCODE').merge(air_clean, on='ADM2_PCODE')
master_named = master_named.merge(crosswalk[['ADM2_PCODE', 'ADM2_EN']], on='ADM2_PCODE')

# normalise indicators
master_named['pm25_norm'] = (master_named['pm25'] - master_named['pm25'].min()) / (master_named['pm25'].max() - master_named['pm25'].min())
master_named['elderly_norm'] = (master_named['elderly'] - master_named['elderly'].min()) / (master_named['elderly'].max() - master_named['elderly'].min())
master_named['children_norm'] = (master_named['children_u5'] - master_named['children_u5'].min()) / (master_named['children_u5'].max() - master_named['children_u5'].min())
master_named['access_norm'] = 1 - (master_named['hospital_access_count'] - master_named['hospital_access_count'].min()) / (master_named['hospital_access_count'].max() - master_named['hospital_access_count'].min())

# updated overlap score with children included
master_named['overlap_score'] = (
    master_named['pm25_norm'] * 0.40 +
    master_named['elderly_norm'] * 0.20 +
    master_named['children_norm'] * 0.10 +
    master_named['access_norm'] * 0.30
).round(3)

master_named = master_named.sort_values('overlap_score', ascending=False)

print(master_named[['ADM2_EN', 'pm25', 'elderly', 'children_u5', 'hospital_access_count', 'overlap_score']])

# save updated version
master_named.to_csv('/drive/MyDrive/philippines_risk_project/master_risk_final.csv', index=False)
print("\nsaved successfully")
```

**What changed in the overlap score weights:**
```
pm25        0.40  (unchanged — strongest CVD evidence)
elderly     0.20  (reduced from 0.30 to make room)
children_u5 0.10  (new — lung development vulnerability)
access_norm 0.30  (unchanged)

In [ ]:
# rebuild master with children_u5 added
vuln_clean = vuln[['ADM2_PCODE', 'elderly', 'rural_pop_perc', 'children_u5']]

acc_clean = acc[['ADM2_PCODE', 'access_pop_hospitals_30min']].copy()
acc_clean['hospital_access_count'] = acc_clean['access_pop_hospitals_30min']
acc_clean = acc_clean[['ADM2_PCODE', 'hospital_access_count']]

adm2_bridged = adm2.merge(manual_crosswalk, on='adm2_pco')
air_clean = adm2_bridged[['ADM2_PCODE', 'pm25', 'no2', 'co']]

master_named = vuln_clean.merge(acc_clean, on='ADM2_PCODE').merge(air_clean, on='ADM2_PCODE')
master_named = master_named.merge(crosswalk[['ADM2_PCODE', 'ADM2_EN']], on='ADM2_PCODE')

master_named['pm25_norm'] = (master_named['pm25'] - master_named['pm25'].min()) / (master_named['pm25'].max() - master_named['pm25'].min())
master_named['elderly_norm'] = (master_named['elderly'] - master_named['elderly'].min()) / (master_named['elderly'].max() - master_named['elderly'].min())
master_named['children_norm'] = (master_named['children_u5'] - master_named['children_u5'].min()) / (master_named['children_u5'].max() - master_named['children_u5'].min())
master_named['access_norm'] = 1 - (master_named['hospital_access_count'] - master_named['hospital_access_count'].min()) / (master_named['hospital_access_count'].max() - master_named['hospital_access_count'].min())

master_named['overlap_score'] = (
    master_named['pm25_norm'] * 0.40 +
    master_named['elderly_norm'] * 0.20 +
    master_named['children_norm'] * 0.10 +
    master_named['access_norm'] * 0.30
).round(3)

master_named = master_named.sort_values('overlap_score', ascending=False)

print(master_named[['ADM2_EN', 'pm25', 'elderly', 'children_u5', 'hospital_access_count', 'overlap_score']])

master_named.to_csv('/drive/MyDrive/philippines_risk_project/master_risk_final.csv', index=False)
print("\nsaved successfully")

In [ ]:
print("vuln_clean shape:", vuln_clean.shape)
print("acc_clean shape:", acc_clean.shape)
print("air_clean shape:", air_clean.shape)
print("master_named shape:", master_named.shape)

In [ ]:
print("adm2 shape:", adm2.shape)
print("adm2 sample:", adm2['adm2_pco'].tolist())
print("manual crosswalk:", manual_crosswalk['adm2_pco'].tolist())

In [ ]:
print("total rows in df:", len(df))
print("unique adm2 codes:", df['adm2_pco'].nunique())

In [ ]:
# check what the raw P-codes look like
print(df['adm4_pcode'].head(5).tolist())
print("P-code length:", df['adm4_pcode'].str.len().value_counts())

In [ ]:
# create permanent lookup table
lookup = pd.DataFrame({
    'aq_station_code': ['PH0155', 'PH0349', 'PH0505', 'PH0630', 'PH0722',
                        'PH0837', 'PH0973', 'PH1043', 'PH1124', 'PH1374',
                        'PH1375', 'PH1376'],
    'ADM2_PCODE':      ['PH01055', 'PH03049', None, 'PH06030', 'PH07022',
                        'PH08037', 'PH09073', 'PH10043', 'PH11024', 'PH13074',
                        'PH13075', 'PH13076'],
    'ADM2_EN':         ['Pangasinan', 'Nueva Ecija', 'UNMATCHED', 'Iloilo', 'Cebu',
                        'Leyte', 'Zamboanga del Sur', 'Misamis Oriental', 'Davao del Sur',
                        'MM 2nd District', 'MM 3rd District', 'MM 4th District']
})

# save permanently to Google Drive
lookup.to_csv('/drive/MyDrive/philippines_risk_project/pcode_lookup.csv', index=False)
print(lookup)
print("\nsaved to Google Drive")

In [ ]:
# slice at 6 characters
df['adm2_pco'] = df['adm4_pcode'].str[:6]
adm2 = df.groupby('adm2_pco')[['pm25', 'no2', 'co']].mean().reset_index()
adm2 = adm2.round(2)
adm2 = adm2.rename(columns={'adm2_pco': 'aq_station_code'})

# join using lookup table
adm2_mapped = adm2.merge(lookup, on='aq_station_code')
adm2_mapped = adm2_mapped[adm2_mapped['ADM2_PCODE'].notna()]
air_clean = adm2_mapped[['ADM2_PCODE', 'pm25', 'no2', 'co']]

print("air_clean shape:", air_clean.shape)
print(air_clean)

In [ ]:
df['adm2_pco'] = df['adm4_pcode'].str[:6]
print("unique 6-char codes:", df['adm2_pco'].unique().tolist())
print("total unique:", df['adm2_pco'].nunique())

In [ ]:
import os
size = os.path.getsize('/drive/MyDrive/philippines_risk_project/climate_air_quality.csv')
print(f"file size: {size / (1024*1024):.1f} MB")

In [ ]:
print("total rows:", len(df))
print("unique station codes:", df['adm2_pco'].unique().tolist())
print("date range:", df['date'].min(), "to", df['date'].max())
print("freq values:", df['freq'].unique().tolist())

In [ ]:
!pip install rasterio geopandas rasterstats --quiet

In [ ]:
import rasterio
import geopandas as gpd
from rasterstats import zonal_stats
import pandas as pd
import json

# load the province boundaries as a geodataframe
provinces = gpd.read_file('/drive/MyDrive/philippines_risk_project/adm2_simplified.geojson')

# run zonal statistcis - average raster pixels within each province boundary
tif_path = '/drive/MyDrive/philippines_risk_project/sdei-global-annual-gwr-pm2-5-modis-misr-seawifs-viirs-aod-v5-gl-04-2022-geotiff.tif'

stats = zonal_stats(
    provinces,
    tif_path,
    stats='mean',
    nodata=-999
)

# add results back to province dataframa
provinces['pm25_satellite'] = [s['mean'] for s in stats]
provinces['pm25_satellite'] = provinces['pm25_satellite'].round(2)

#preview
print(provinces[['ADM2_PCODE', 'AMD2_EN', 'pm25_satellite']].head(10))

#preview
print(provinces[['ADM2_PCODE', 'ADM2_EN', 'pm25_satellite']].head(10))
print("n\notal provinces with PM2.5 data:", provinces['pm25_satellite'].notna().sum())


In [ ]:
print(provinces[['ADM2_PCODE', 'ADM2_EN', 'pm25_satellite']].head(10))
print("\ntotal provinces with PM2.5 data:", provinces['pm25_satellite'].notna().sum())

In [ ]:
import rasterio
with rasterio.open(tif_path) as src:
  print("nodata value:", src.nodata)
  print("data type:", src.count)

  # read a small sample of actual values
  data = src.read(1)
  print("min value in raster:", data.min())
  print("max value in raster:", data.max())
  print("sample values:", data[data > 0][:10])


In [ ]:
nodata_value = -3.3999999521443642e+38

stats = zonal_stats(
    provinces,
    tif_path,
    stats='mean',
    nodata=nodata_value
)

# add results back to province dataframe
provinces['pm25_satellite'] = [s['mean'] for s in stats]
provinces['pm25_satellite'] = provinces['pm25_satellite'].round(2)

# preview
print(provinces[['ADM2_PCODE', 'ADM2_EN', 'pm25_satellite']].head(15))
print("\ntotal provinces with PM2.5 data:", provinces['pm25_satellite'].notna().sum())
print("min PM2.5:", provinces['pm25_satellite'].min())
print("max PM2.5:", provinces['pm25_satellite'].max())


In [ ]:
print(provinces[['ADM2_EN', 'pm25_satellite']].sort_values('pm25_satellite', ascending=False).head(5))


In [ ]:
# merge satellite PM2.5 with vulnerability and access data
pm25_clean = provinces[['ADM2_PCODE', 'pm25_satellite']].copy()
pm25_clean = pm25_clean.rename(columns={'pm25_satellite': 'pm25'})

#rebuild vulnerability and access
vuln_clean = vuln[['ADM2_PCODE', 'elderly', 'rural_pop_perc', 'children_u5']]
acc_clean = acc[['ADM2_PCODE', 'access_pop_hospitals_30min']].copy()
acc_clean = acc_clean.rename(columns={'access_pop_hospitals_30min': 'hospital_access_count'})

# merge everything
master_full = pm25_clean.merge(vuln_clean, on='ADM2_PCODE').merge(acc_clean, on='ADM2_PCODE')

print(master_full.shape)
print(master_full.head(5))

In [ ]:
# normalise each indicator to 0-1 scale
master_full['pm25_norm'] = (master_full['pm25'] - master_full['pm25'].min()) / (master_full['pm25'].max() - master_full['pm25'].min())
master_full['elderly_norm'] = (master_full['elderly'] - master_full['elderly'].min()) / (master_full['elderly'].max() - master_full['elderly'].min())
master_full['children_norm'] = (master_full['children_u5'] - master_full['children_u5'].min()) / (master_full['children_u5'].max() - master_full['children_u5'].min())
master_full['access_norm'] = 1 - (master_full['hospital_access_count'] - master_full['hospital_access_count'].min()) / (master_full['hospital_access_count'].max() - master_full['hospital_access_count'].min())

# weighted overlap score
master_full['overlap_score'] = (
    master_full['pm25_norm'] * 0.40 +
    master_full['elderly_norm'] * 0.20 +
    master_full['children_norm'] * 0.10 +
    master_full['access_norm'] * 0.30
).round(3)

# sort by highest risk first
master_full = master_full.sort_values('overlap_score', ascending=False)

# preview top 10
print(master_full[['ADM2_EN', 'pm25', 'elderly', 'children_u5', 'hospital_access_count', 'overlap_score']].head(10))

# save to Google Drive
master_full.to_csv('/drive/MyDrive/philippines_risk_project/master_risk_final_v2.csv', index=False)
print("\nsaved successfully")

In [ ]:
print(master_full.columns.tolist())

In [ ]:
# add province names back
master_full = master_full.merge(crosswalk[['ADM2_PCODE', 'ADM2_EN']], on='ADM2_PCODE')

# preview top 10
print(master_full[['ADM2_EN', 'pm25', 'elderly', 'children_u5', 'hospital_access_count', 'overlap_score']].head(10))

# save to Google Drive
master_full.to_csv('/drive/MyDrive/philippines_risk_project/master_risk_final_v2.csv', index=False)
print("\nsaved successfully")

In [ ]:
# merge overlap scores into the provinces geodataframe
provinces_final = provinces.merge(
    master_full[['ADM2_PCODE', 'overlap_score', 'pm25', 'elderly',
                 'children_u5', 'rural_pop_perc', 'hospital_access_count', 'ADM2_EN']],
    on='ADM2_PCODE'
)

# save as new geojson
provinces_final.to_file('/drive/MyDrive/philippines_risk_project/philippines_risk_v2.geojson',
                         driver='GeoJSON')

print("provinces in final geodataframe:", len(provinces_final))
print("saved successfully")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json

# load the new geojson
with open('/drive/MyDrive/philippines_risk_project/philippines_risk_v2.geojson') as f:
    geojson_v2 = json.load(f)

# sort for bar chart
df_sorted = master_full.sort_values('overlap_score', ascending=True)

# shorten long names
name_map = {
    'Metropolitan Manila First District': 'MM 1st District',
    'Metropolitan Manila Second District': 'MM 2nd District',
    'Metropolitan Manila Third District': 'MM 3rd District',
    'Metropolitan Manila Fourth District': 'MM 4th District',
    'Cotabato (North Cotabato)': 'North Cotabato',
    'Samar (Western Samar)': 'Western Samar',
    'Davao de Oro (Compostela Valley)': 'Davao de Oro',
    'City of Isabela (not a province)': 'Isabela City',
    'Special Geographic Area': 'BARMM Spec. Area'
}

master_full['display_name'] = master_full['ADM2_EN'].replace(name_map)
df_sorted['display_name'] = df_sorted['ADM2_EN'].replace(name_map)

fig3 = make_subplots(
    rows=1, cols=2,
    column_widths=[0.55, 0.45],
    specs=[[{'type': 'choropleth'}, {'type': 'bar'}]],
    subplot_titles=['Health risk overlap by province', 'Ranked overlap score']
)

fig3.add_trace(
    go.Choropleth(
        geojson=geojson_v2,
        locations=master_full['ADM2_PCODE'],
        z=master_full['overlap_score'],
        featureidkey='properties.ADM2_PCODE',
        colorscale='RdYlGn_r',
        showscale=True,
        text=master_full['display_name'],
        customdata=master_full[['pm25','elderly','children_u5','rural_pop_perc','hospital_access_count']].values,
        hovertemplate='<b>%{text}</b><br>Health risk overlap score: %{z}<br>PM2.5 (satellite, 2022): %{customdata[0]} ug/m3<br>Elderly population (60+): %{customdata[1]:,}<br>Children under 5: %{customdata[2]:,}<br>Rural population: %{customdata[3]}%<br>Est. pop within 30min of hospital: %{customdata[4]:,}<extra></extra>'
    ),
    row=1, col=1
)

fig3.add_trace(
    go.Bar(
        x=df_sorted['overlap_score'],
        y=df_sorted['display_name'],
        orientation='h',
        marker_color=df_sorted['overlap_score'],
        marker_colorscale='RdYlGn_r',
        text=df_sorted['overlap_score'],
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>Score: %{x}<extra></extra>'
    ),
    row=1, col=2
)

fig3.update_geos(fitbounds='locations', visible=False)
fig3.update_layout(
    title=dict(
        text='Philippines Climate-Health Risk Index — Air Pollution, Vulnerability & Healthcare Access Gap<br><sup>82 provinces · NASA satellite PM2.5 2022 · HDX vulnerability & access data · Higher score = greater compounded risk</sup>',
        x=0.5,
        xanchor='center'
    ),
    height=700,
    showlegend=False
)

fig3.write_html('/drive/MyDrive/philippines_risk_project/philippines_health_risk_dashboard_final.html')
fig3.show()
print("saved successfully")

In [ ]:
fig_map.show()

In [ ]:
fig_map.update_layout(
    mapbox=dict(
        accesstoken=mapbox_token,
        style='mapbox://styles/mapbox/satellite-streets-v12',
        center=dict(lat=12.5, lon=122.5),
        zoom=4.5
    ),
    margin=dict(l=0, r=0, t=80, b=0),
    height=650,
    title=dict(
        text='Philippines Climate-Health Risk Index<br><sup>Air Pollution · Vulnerability · Healthcare Access Gap · NASA satellite PM2.5 2022</sup>',
        x=0.5,
        xanchor='center',
        font=dict(size=16)
    )
)

# update colorbar to be more visible
fig_map.update_traces(
    colorbar=dict(
        title=dict(
            text='Risk score',
            font=dict(size=14)
        ),
        thickness=25,
        len=0.8,
        x=1.0,
        tickfont=dict(size=13),
        tickvals=[
            master_full['overlap_score'].min(),
            master_full['overlap_score'].quantile(0.25),
            master_full['overlap_score'].mean(),
            master_full['overlap_score'].quantile(0.75),
            master_full['overlap_score'].max()
        ],
        ticktext=['Low', 'Low-Med', 'Medium', 'Med-High', 'High']
    )
)

fig_map.write_html('/drive/MyDrive/philippines_risk_project/philippines_risk_map_final.html')
fig_map.show()
print("saved successfully")

In [ ]:
fig_map.update_traces(
    colorbar=dict(
        title=dict(
            text='Health risk<br>overlap score',
            font=dict(size=13)
        ),
        thickness=20,
        len=0.75,
        x=0.98,
        tickfont=dict(size=12),
        tickvals=[
            master_full['overlap_score'].min(),
            master_full['overlap_score'].quantile(0.25),
            master_full['overlap_score'].median(),
            master_full['overlap_score'].quantile(0.75),
            master_full['overlap_score'].max()
        ],
        ticktext=[
            str(round(master_full['overlap_score'].min(), 2)),
            str(round(master_full['overlap_score'].quantile(0.25), 2)),
            str(round(master_full['overlap_score'].median(), 2)),
            str(round(master_full['overlap_score'].quantile(0.75), 2)),
            str(round(master_full['overlap_score'].max(), 2))
        ],
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(0,0,0,0.2)',
        borderwidth=1
    )
)

fig_map.show()

In [ ]:
fig_map.update_traces(
    colorbar=dict(
        title=dict(
            text='Health risk<br>overlap score<br><br> ',
            font=dict(size=13),
            side='top'
        ),
        thickness=20,
        len=0.75,
        x=0.98,
        tickfont=dict(size=12),
        tickvals=[
            master_full['overlap_score'].min(),
            master_full['overlap_score'].quantile(0.25),
            master_full['overlap_score'].median(),
            master_full['overlap_score'].quantile(0.75),
            master_full['overlap_score'].max()
        ],
        ticktext=[
            str(round(master_full['overlap_score'].min(), 2)),
            str(round(master_full['overlap_score'].quantile(0.25), 2)),
            str(round(master_full['overlap_score'].median(), 2)),
            str(round(master_full['overlap_score'].quantile(0.75), 2)),
            str(round(master_full['overlap_score'].max(), 2))
        ],
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(0,0,0,0.2)',
        borderwidth=1
    )
)

fig_map.show()

In [ ]:
map_html = fig_map.to_html(full_html=False, include_plotlyjs=True)
bar_html = fig_bar.to_html(full_html=False, include_plotlyjs=False)

dashboard = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Philippines Climate-Health Risk Index</title>
    <style>
        * {{ box-sizing: border-box; margin: 0; padding: 0; font-family: Arial, sans-serif; }}
        body {{ background: #f4f4f4; }}
        .header {{
            background: #1B5E8A;
            color: white;
            padding: 20px 30px;
        }}
        .header h1 {{ font-size: 20px; font-weight: 600; margin-bottom: 6px; }}
        .header p {{ font-size: 13px; opacity: 0.85; }}
        .tabs {{
            display: flex;
            background: #ffffff;
            border-bottom: 2px solid #1B5E8A;
            padding: 0 30px;
        }}
        .tab {{
            padding: 12px 24px;
            cursor: pointer;
            font-size: 14px;
            color: #555;
            border-bottom: 3px solid transparent;
            margin-bottom: -2px;
        }}
        .tab.active {{
            color: #1B5E8A;
            font-weight: 600;
            border-bottom: 3px solid #1B5E8A;
        }}
        .panel {{ display: none; padding: 20px 30px; }}
        .panel.active {{ display: block; }}
        .map-container {{
            background: white;
            border-radius: 8px;
            overflow: hidden;
            box-shadow: 0 1px 4px rgba(0,0,0,0.1);
        }}
        .bar-container {{
            background: white;
            border-radius: 8px;
            overflow: hidden;
            box-shadow: 0 1px 4px rgba(0,0,0,0.1);
        }}
        .meta {{
            font-size: 12px;
            color: #888;
            margin-top: 10px;
            padding: 0 4px;
        }}
    </style>
</head>
<body>
    <div class="header">
        <h1>Philippines Climate-Health Risk Index</h1>
        <p>Air Pollution · Population Vulnerability · Healthcare Access Gap · 88 Provinces · NASA Satellite PM2.5 2022</p>
    </div>
    <div class="tabs">
        <div class="tab active" onclick="showTab('map', this)">Risk Map</div>
        <div class="tab" onclick="showTab('bar', this)">Province Rankings</div>
    </div>
    <div id="tab-map" class="panel active">
        <div class="map-container">
            {map_html}
        </div>
        <p class="meta">Hover over any province to see detailed indicators. Zoom and pan freely. Red = highest compounded risk, Green = lowest.</p>
    </div>
    <div id="tab-bar" class="panel">
        <div class="bar-container">
            {bar_html}
        </div>
        <p class="meta">All 88 provinces ranked by health risk overlap score. Hover over any bar for full indicator breakdown.</p>
    </div>
    <script>
        function showTab(name, el) {{
            document.querySelectorAll('.panel').forEach(p => p.classList.remove('active'));
            document.querySelectorAll('.tab').forEach(t => t.classList.remove('active'));
            document.getElementById('tab-' + name).classList.add('active');
            el.classList.add('active');
        }}
    </script>
</body>
</html>
"""

with open('/drive/MyDrive/philippines_risk_project/philippines_health_risk_dashboard_final.html', 'w') as f:
    f.write(dashboard)

print("dashboard saved successfully")

In [ ]:
map_html = fig_map.to_html(full_html=False, include_plotlyjs=True)
bar_html = fig_bar.to_html(full_html=False, include_plotlyjs=False)

dashboard = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Philippines Climate-Health Risk Index</title>
    <style>
        * {{ box-sizing: border-box; margin: 0; padding: 0; font-family: Arial, sans-serif; }}
        body {{ background: #f4f4f4; }}
        .header {{
            background: #1B5E8A;
            color: white;
            padding: 20px 30px;
        }}
        .header h1 {{ font-size: 20px; font-weight: 600; margin-bottom: 6px; }}
        .header p {{ font-size: 13px; opacity: 0.85; }}
        .tabs {{
            display: flex;
            background: #ffffff;
            border-bottom: 2px solid #1B5E8A;
            padding: 0 30px;
        }}
        .tab {{
            padding: 12px 24px;
            cursor: pointer;
            font-size: 14px;
            color: #555;
            border-bottom: 3px solid transparent;
            margin-bottom: -2px;
        }}
        .tab.active {{
            color: #1B5E8A;
            font-weight: 600;
            border-bottom: 3px solid #1B5E8A;
        }}
        .panel {{ display: none; padding: 20px 30px; }}
        .panel.active {{ display: block; }}
        .map-container {{
            background: white;
            border-radius: 8px;
            overflow: hidden;
            box-shadow: 0 1px 4px rgba(0,0,0,0.1);
            position: relative;
        }}
        .zoom-buttons {{
    position: absolute;
    bottom: 30px;
    left: 20px;
    z-index: 999;
    display: flex;
    flex-direction: column;
    gap: 4px;

        }}
        .zoom-btn {{
            width: 32px;
            height: 32px;
            background: white;
            border: 1px solid #ccc;
            border-radius: 4px;
            font-size: 18px;
            cursor: pointer;
            display: flex;
            align-items: center;
            justify-content: center;
            box-shadow: 0 1px 3px rgba(0,0,0,0.2);
            color: #333;
        }}
        .zoom-btn:hover {{ background: #f0f0f0; }}
        .bar-container {{
            background: white;
            border-radius: 8px;
            overflow: hidden;
            box-shadow: 0 1px 4px rgba(0,0,0,0.1);
        }}
        .meta {{
            font-size: 12px;
            color: #888;
            margin-top: 10px;
            padding: 0 4px;
        }}
        .methodology {{
            background: white;
            border-radius: 8px;
            padding: 16px 20px;
            margin-top: 16px;
            box-shadow: 0 1px 4px rgba(0,0,0,0.1);
            font-size: 13px;
            color: #444;
            line-height: 1.7;
        }}
        .methodology h3 {{
            font-size: 14px;
            color: #1B5E8A;
            margin-bottom: 10px;
            font-weight: 600;
        }}
        .indicator-row {{
            display: flex;
            gap: 12px;
            flex-wrap: wrap;
            margin-top: 10px;
        }}
        .indicator {{
            background: #f0f5fa;
            border-radius: 6px;
            padding: 10px 14px;
            flex: 1;
            min-width: 180px;
        }}
        .indicator-name {{
            font-weight: 600;
            color: #1B5E8A;
            font-size: 13px;
            margin-bottom: 4px;
        }}
        .indicator-desc {{
            font-size: 12px;
            color: #666;
        }}
        .indicator-weight {{
            font-size: 12px;
            font-weight: 600;
            color: #444;
            margin-top: 4px;
        }}
    </style>
</head>
<body>
    <div class="header">
        <h1>Philippines Climate-Health Risk Index</h1>
        <p>Air Pollution · Population Vulnerability · Healthcare Access Gap · 88 Provinces · NASA Satellite PM2.5 2022</p>
    </div>
    <div class="tabs">
        <div class="tab active" onclick="showTab('map', this)">Risk Map</div>
        <div class="tab" onclick="showTab('bar', this)">Province Rankings</div>
        <div class="tab" onclick="showTab('about', this)">About this index</div>
    </div>

    <div id="tab-map" class="panel active">
        <div class="map-container">
            {map_html}
            <div class="zoom-buttons">
                <button class="zoom-btn" onclick="zoomIn()">+</button>
                <button class="zoom-btn" onclick="zoomOut()">−</button>
            </div>
        </div>
        <p class="meta">Hover over any province to see detailed indicators. Scroll to zoom, click and drag to pan. Red = highest compounded risk, Green = lowest.</p>
    </div>

    <div id="tab-bar" class="panel">
        <div class="bar-container">
            {bar_html}
        </div>
        <p class="meta">All 88 provinces ranked by health risk overlap score. Hover over any bar for full indicator breakdown.</p>
    </div>

    <div id="tab-about" class="panel">
        <div class="methodology">
            <h3>How the health risk overlap score is calculated</h3>
            <p>The overlap score is a composite index ranging from 0 to 1 that combines three dimensions of health risk.
            A higher score means a province has greater compounded risk across all three dimensions simultaneously —
            not just high risk in one area but overlapping vulnerabilities across pollution, population, and healthcare access.</p>
            <div class="indicator-row">
                <div class="indicator">
                    <div class="indicator-name">Air pollution — PM2.5</div>
                    <div class="indicator-desc">Annual mean fine particulate matter concentration (μg/m³) derived from NASA satellite data (MODIS, MISR, VIIRS, 2022). PM2.5 has the strongest direct evidence linking air pollution to cardiovascular and respiratory mortality.</div>
                    <div class="indicator-weight">Weight: 40%</div>
                </div>
                <div class="indicator">
                    <div class="indicator-name">Population vulnerability</div>
                    <div class="indicator-desc">Combination of elderly population (60+) and children under 5 — the two age groups most physiologically vulnerable to air pollution and least able to cope with health shocks. Source: HDX / Philippine Statistics Authority.</div>
                    <div class="indicator-weight">Elderly weight: 20% · Children weight: 10%</div>
                </div>
                <div class="indicator">
                    <div class="indicator-name">Healthcare access gap</div>
                    <div class="indicator-desc">Estimated population within 30 minutes of a hospital, inverted so that lower access produces a higher risk score. Provinces where fewer people can reach care quickly face greater consequences from pollution-related illness. Source: HDX / OCHA.</div>
                    <div class="indicator-weight">Weight: 30%</div>
                </div>
            </div>
            <p style="margin-top:14px"><b>Formula:</b> Overlap score = (PM2.5 × 0.40) + (Elderly × 0.20) + (Children × 0.10) + (Access gap × 0.30)</p>
            <p style="margin-top:8px"><b>Data sources:</b> NASA SEDAC Global Annual PM2.5 Grids V5.GL.04 (2022) · HDX PHL Risk Assessment Indicators · Philippine Statistics Authority · OCHA</p>
            <p style="margin-top:8px"><b>Limitation:</b> Air quality data reflects satellite-derived estimates, not ground measurements. Hospital access reflects modelled travel times, not actual utilisation rates.</p>
        </div>
    </div>

    <script>
        function showTab(name, el) {{
            document.querySelectorAll('.panel').forEach(p => p.classList.remove('active'));
            document.querySelectorAll('.tab').forEach(t => t.classList.remove('active'));
            document.getElementById('tab-' + name).classList.add('active');
            el.classList.add('active');
        }}

        var currentZoom = 4.5;
        function zoomIn() {{
            currentZoom = Math.min(currentZoom + 0.5, 12);
            var mapDiv = document.querySelector('.js-plotly-plot');
            if (mapDiv) {{
                Plotly.relayout(mapDiv, {{'mapbox.zoom': currentZoom}});
            }}
        }}
        function zoomOut() {{
            currentZoom = Math.max(currentZoom - 0.5, 1);
            var mapDiv = document.querySelector('.js-plotly-plot');
            if (mapDiv) {{
                Plotly.relayout(mapDiv, {{'mapbox.zoom': currentZoom}});
            }}
        }}
    </script>
</body>
</html>
"""

with open('/drive/MyDrive/philippines_risk_project/philippines_health_risk_dashboard_final.html', 'w') as f:
    f.write(dashboard)

print("dashboard saved successfully")

fig_map.show()



In [ ]:
fig_map.update_traces(
    colorbar=dict(
        title=dict(
            text='Health risk<br>overlap score<br><br><br> ',
            font=dict(size=13),
            side='top'
        ),
        thickness=20,
        len=0.75,
        x=0.98,
        tickfont=dict(size=12),
        tickvals=[
            master_full['overlap_score'].min(),
            master_full['overlap_score'].quantile(0.25),
            master_full['overlap_score'].median(),
            master_full['overlap_score'].quantile(0.75),
            master_full['overlap_score'].max()
        ],
        ticktext=[
            str(round(master_full['overlap_score'].min(), 2)),
            str(round(master_full['overlap_score'].quantile(0.25), 2)),
            str(round(master_full['overlap_score'].median(), 2)),
            str(round(master_full['overlap_score'].quantile(0.75), 2)),
            str(round(master_full['overlap_score'].max(), 2))
        ],
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(0,0,0,0.2)',
        borderwidth=1
    )
)

fig_map.show()

In [ ]:
fig_map.update_traces(
    colorbar=dict(
        title=dict(
            text='Health risk overlap score',
            font=dict(size=13),
            side='bottom'
        ),
        thickness=20,
        len=0.75,
        x=0.98,
        tickfont=dict(size=12),
        tickvals=[
            master_full['overlap_score'].min(),
            master_full['overlap_score'].quantile(0.25),
            master_full['overlap_score'].median(),
            master_full['overlap_score'].quantile(0.75),
            master_full['overlap_score'].max()
        ],
        ticktext=[
            str(round(master_full['overlap_score'].min(), 2)),
            str(round(master_full['overlap_score'].quantile(0.25), 2)),
            str(round(master_full['overlap_score'].median(), 2)),
            str(round(master_full['overlap_score'].quantile(0.75), 2)),
            str(round(master_full['overlap_score'].max(), 2))
        ],
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(0,0,0,0.2)',
        borderwidth=1
    )
)

fig_map.show()

In [ ]:
# remove title from colorbar
fig_map.update_traces(
    colorbar=dict(
        title=dict(text=''),
        thickness=20,
        len=0.75,
        x=0.98,
        tickfont=dict(size=12),
        tickvals=[
            master_full['overlap_score'].min(),
            master_full['overlap_score'].quantile(0.25),
            master_full['overlap_score'].median(),
            master_full['overlap_score'].quantile(0.75),
            master_full['overlap_score'].max()
        ],
        ticktext=[
            str(round(master_full['overlap_score'].min(), 2)),
            str(round(master_full['overlap_score'].quantile(0.25), 2)),
            str(round(master_full['overlap_score'].median(), 2)),
            str(round(master_full['overlap_score'].quantile(0.75), 2)),
            str(round(master_full['overlap_score'].max(), 2))
        ],
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(0,0,0,0.2)',
        borderwidth=1
    )
)

# add title as annotation above colorbar
fig_map.update_layout(
    annotations=[
        dict(
            x=1.02,
            y=1.02,
            xref='paper',
            yref='paper',
            text='<b>Health risk<br>overlap score</b>',
            showarrow=False,
            font=dict(size=13, color='#333'),
            align='left'
        )
    ]
)

fig_map.show()

In [ ]:
fig_map.update_layout(
    annotations=[
        dict(
            x=1.08,
            y=0.98,
            xref='paper',
            yref='paper',
            text='<b>Health risk<br>overlap score</b>',
            showarrow=False,
            font=dict(size=12, color='#333'),
            align='left',
            bgcolor='rgba(255,255,255,0.85)',
            bordercolor='rgba(0,0,0,0.2)',
            borderwidth=1,
            borderpad=4
        )
    ],
    margin=dict(l=0, r=120, t=80, b=0)
)

fig_map.show()

In [ ]:
fig_map.update_layout(
    annotations=[
        dict(
            x=1.13,
            y=0.98,
            xref='paper',
            yref='paper',
            text='<b>Health risk<br>overlap score</b>',
            showarrow=False,
            font=dict(size=12, color='#333'),
            align='center',
            bgcolor='rgba(255,255,255,0.85)',
            bordercolor='rgba(0,0,0,0.2)',
            borderwidth=1,
            borderpad=4
        )
    ],
    margin=dict(l=0, r=150, t=80, b=0)
)

fig_map.show()

In [ ]:
fig_map.update_traces(
    colorbar=dict(
        title=dict(
            text='Health risk<br>overlap score',
            font=dict(size=12),
            side='top'
        ),
        thickness=20,
        len=0.75,
        x=0.98,
        y=0.5,
        tickfont=dict(size=12),
        tickvals=[
            master_full['overlap_score'].min(),
            master_full['overlap_score'].quantile(0.25),
            master_full['overlap_score'].median(),
            master_full['overlap_score'].quantile(0.75),
            master_full['overlap_score'].max()
        ],
        ticktext=[
            str(round(master_full['overlap_score'].min(), 2)),
            str(round(master_full['overlap_score'].quantile(0.25), 2)),
            str(round(master_full['overlap_score'].median(), 2)),
            str(round(master_full['overlap_score'].quantile(0.75), 2)),
            str(round(master_full['overlap_score'].max(), 2))
        ],
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(0,0,0,0.2)',
        borderwidth=1,
        ticks='outside',
        ticklabelposition='outside'
    )
)

# remove annotation
fig_map.update_layout(annotations=[])

fig_map.show()

In [ ]:
fig_map.update_traces(
    colorbar=dict(
        title=dict(
            text='Health risk<br>overlap score',
            font=dict(size=12),
            side='top'
        ),
        thickness=20,
        len=0.75,
        x=0.98,
        tickfont=dict(size=12),
        tickvals=[
            master_full['overlap_score'].min(),
            master_full['overlap_score'].quantile(0.25),
            master_full['overlap_score'].median(),
            master_full['overlap_score'].quantile(0.75),
            master_full['overlap_score'].max()
        ],
        ticktext=[
            str(round(master_full['overlap_score'].min(), 2)),
            str(round(master_full['overlap_score'].quantile(0.25), 2)),
            str(round(master_full['overlap_score'].median(), 2)),
            str(round(master_full['overlap_score'].quantile(0.75), 2)),
            str(round(master_full['overlap_score'].max(), 2))
        ],
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(0,0,0,0.2)',
        borderwidth=1
    )
)

fig_map.update_layout(annotations=[])
fig_map.update_layout(margin=dict(l=0, r=0, t=80, b=0))

fig_map.show()

In [ ]:
map_html = fig_map.to_html(full_html=False, include_plotlyjs=True, config={'scrollZoom': True, 'displayModeBar': False})
bar_html = fig_bar.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': False})

province_json = df_sorted[['display_name', 'overlap_score']].to_json(orient='records')

dashboard = open('/drive/MyDrive/philippines_risk_project/philippines_health_risk_dashboard_final.html', 'w')
dashboard.write(f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Philippines Climate-Health Risk Index</title>
    <style>
        * {{ box-sizing: border-box; margin: 0; padding: 0; font-family: Arial, sans-serif; }}
        body {{ background: #f4f4f4; }}
        .header {{ background: #1B5E8A; color: white; padding: 20px 30px; }}
        .header h1 {{ font-size: 20px; font-weight: 600; margin-bottom: 6px; }}
        .header p {{ font-size: 13px; opacity: 0.85; }}
        .tabs {{ display: flex; background: #ffffff; border-bottom: 2px solid #1B5E8A; padding: 0 30px; }}
        .tab {{ padding: 12px 24px; cursor: pointer; font-size: 14px; color: #555; border-bottom: 3px solid transparent; margin-bottom: -2px; transition: all 0.2s; }}
        .tab.active {{ background: #1B5E8A; color: white; font-weight: 600; border-bottom: 3px solid #1B5E8A; }}
        .panel {{ display: none; padding: 20px 30px; }}
        .panel.active {{ display: block; }}
        .map-container {{ background: white; border-radius: 8px; overflow: hidden; box-shadow: 0 1px 4px rgba(0,0,0,0.1); position: relative; }}
        .zoom-buttons {{ position: absolute; bottom: 30px; left: 20px; z-index: 999; display: flex; flex-direction: column; gap: 4px; }}
        .zoom-btn {{ width: 32px; height: 32px; background: white; border: 1px solid #ccc; border-radius: 4px; font-size: 18px; cursor: pointer; display: flex; align-items: center; justify-content: center; box-shadow: 0 1px 3px rgba(0,0,0,0.2); color: #333; }}
        .zoom-btn:hover {{ background: #f0f0f0; }}
        .bar-container {{ background: white; border-radius: 8px; overflow: hidden; box-shadow: 0 1px 4px rgba(0,0,0,0.1); }}
        .search-wrap {{ padding: 16px 20px 8px; background: white; border-radius: 8px 8px 0 0; display: flex; align-items: center; gap: 12px; border-bottom: 0.5px solid #eee; }}
        .search-input {{ flex: 1; padding: 8px 14px; border: 1px solid #ccc; border-radius: 6px; font-size: 14px; outline: none; max-width: 360px; }}
        .search-input:focus {{ border-color: #1B5E8A; }}
        .search-label {{ font-size: 13px; color: #666; }}
        .match-count {{ font-size: 13px; color: #1B5E8A; font-weight: 600; min-width: 120px; }}
        .clear-btn {{ padding: 6px 12px; background: #f0f0f0; border: 1px solid #ccc; border-radius: 6px; font-size: 13px; cursor: pointer; color: #555; }}
        .clear-btn:hover {{ background: #e0e0e0; }}
        .meta {{ font-size: 12px; color: #888; margin-top: 10px; padding: 0 4px; }}
        .methodology {{ background: white; border-radius: 8px; padding: 16px 20px; margin-top: 16px; box-shadow: 0 1px 4px rgba(0,0,0,0.1); font-size: 13px; color: #444; line-height: 1.7; }}
        .methodology h3 {{ font-size: 14px; color: #1B5E8A; margin-bottom: 10px; font-weight: 600; }}
        .indicator-row {{ display: flex; gap: 12px; flex-wrap: wrap; margin-top: 10px; }}
        .indicator {{ background: #f0f5fa; border-radius: 6px; padding: 10px 14px; flex: 1; min-width: 180px; }}
        .indicator-name {{ font-weight: 600; color: #1B5E8A; font-size: 13px; margin-bottom: 4px; }}
        .indicator-desc {{ font-size: 12px; color: #666; }}
        .indicator-weight {{ font-size: 12px; font-weight: 600; color: #444; margin-top: 4px; }}
        .footer {{ background: #1B5E8A; color: rgba(255,255,255,0.7); font-size: 12px; padding: 12px 30px; margin-top: 30px; }}
    </style>
</head>
<body>
    <div class="header">
        <h1>Philippines Climate-Health Risk Index</h1>
        <p>Air Pollution · Population Vulnerability · Healthcare Access Gap · 88 Provinces · NASA Satellite PM2.5 2022</p>
    </div>
    <div class="tabs">
        <div class="tab active" onclick="showTab('map', this)">Risk Map</div>
        <div class="tab" onclick="showTab('bar', this)">Province Rankings</div>
        <div class="tab" onclick="showTab('about', this)">About this index</div>
    </div>
    <div id="tab-map" class="panel active">
        <div class="map-container">
            {map_html}
            <div class="zoom-buttons">
                <button class="zoom-btn" onclick="zoomIn()">+</button>
                <button class="zoom-btn" onclick="zoomOut()">−</button>
            </div>
        </div>
        <p class="meta">Hover over any province to see detailed indicators. Scroll to zoom, click and drag to pan. Red = highest compounded risk, Green = lowest.</p>
    </div>
    <div id="tab-bar" class="panel">
        <div class="search-wrap">
            <span class="search-label">Search province:</span>
            <input class="search-input" id="province-search" placeholder="Type a province name e.g. Cebu, Benguet..." oninput="highlightProvince(this.value)">
            <span class="match-count" id="match-count"></span>
            <button class="clear-btn" onclick="clearSearch()">Clear</button>
        </div>
        <div class="bar-container" id="bar-chart-container">
            {bar_html}
        </div>
        <p class="meta">All 88 provinces ranked by health risk overlap score. Search to highlight a province. Hover over any bar for full indicator breakdown.</p>
    </div>
    <div id="tab-about" class="panel">
        <div class="methodology">
            <h3>How the health risk overlap score is calculated</h3>
            <p>The overlap score is a composite index ranging from 0 to 1 that combines three dimensions of health risk. A higher score means a province has greater compounded risk across all three dimensions simultaneously.</p>
            <div class="indicator-row">
                <div class="indicator">
                    <div class="indicator-name">Air pollution — PM2.5</div>
                    <div class="indicator-desc">Annual mean fine particulate matter (μg/m³) from NASA satellite data (MODIS, MISR, VIIRS, 2022). Strongest direct evidence linking air pollution to cardiovascular mortality.</div>
                    <div class="indicator-weight">Weight: 40%</div>
                </div>
                <div class="indicator">
                    <div class="indicator-name">Population vulnerability</div>
                    <div class="indicator-desc">Elderly population (60+) and children under 5 — the two age groups most vulnerable to air pollution and least able to cope with health shocks. Source: HDX / PSA.</div>
                    <div class="indicator-weight">Elderly: 20% · Children: 10%</div>
                </div>
                <div class="indicator">
                    <div class="indicator-name">Healthcare access gap</div>
                    <div class="indicator-desc">Estimated population within 30 minutes of a hospital, inverted so lower access = higher risk. Source: HDX / OCHA.</div>
                    <div class="indicator-weight">Weight: 30%</div>
                </div>
            </div>
            <p style="margin-top:14px"><b>Formula:</b> Overlap score = (PM2.5 × 0.40) + (Elderly × 0.20) + (Children × 0.10) + (Access gap × 0.30)</p>
            <p style="margin-top:8px"><b>Data sources:</b> NASA SEDAC V5.GL.04 (2022) · HDX PHL Risk Assessment · PSA · OCHA</p>
            <p style="margin-top:8px"><b>Limitations:</b> PM2.5 reflects satellite estimates not ground measurements. Hospital access reflects modelled travel times not actual utilisation.</p>
        </div>
    </div>
    <div class="footer">
        PM2.5 data: NASA SEDAC 2022 · Vulnerability data: PSA 2020 · Healthcare access: OCHA · Built with Python, Plotly and Mapbox
    </div>
    <script>
        function showTab(name, el) {{
            document.querySelectorAll('.panel').forEach(p => p.classList.remove('active'));
            document.querySelectorAll('.tab').forEach(t => t.classList.remove('active'));
            document.getElementById('tab-' + name).classList.add('active');
            el.classList.add('active');
        }}
        var currentZoom = 4.5;
        function zoomIn() {{
            currentZoom = Math.min(currentZoom + 0.5, 12);
            var mapDiv = document.querySelector('.js-plotly-plot');
            if (mapDiv) Plotly.relayout(mapDiv, {{'mapbox.zoom': currentZoom}});
        }}
        function zoomOut() {{
            currentZoom = Math.max(currentZoom - 0.5, 1);
            var mapDiv = document.querySelector('.js-plotly-plot');
            if (mapDiv) Plotly.relayout(mapDiv, {{'mapbox.zoom': currentZoom}});
        }}
        var provinceData = {province_json};
        function highlightProvince(query) {{
            var barDiv = document.getElementById('bar-chart-container').querySelector('.js-plotly-plot');
            if (!barDiv) return;
            var matchCount = document.getElementById('match-count');
            if (!query || query.length < 2) {{
                Plotly.restyle(barDiv, {{'marker.opacity': [provinceData.map(function() {{ return 1; }})], 'marker.line.width': [provinceData.map(function() {{ return 0; }})]}});
                matchCount.textContent = '';
                return;
            }}
            var q = query.toLowerCase();
            var matches = 0;
            var opacities = provinceData.map(function(d) {{ var m = d.display_name.toLowerCase().indexOf(q) !== -1; if (m) matches++; return m ? 1 : 0.15; }});
            var lineWidths = provinceData.map(function(d) {{ return d.display_name.toLowerCase().indexOf(q) !== -1 ? 2 : 0; }});
            Plotly.restyle(barDiv, {{'marker.opacity': [opacities], 'marker.line.width': [lineWidths], 'marker.line.color': ['#1B5E8A']}});
            matchCount.textContent = matches === 0 ? 'No match found' : matches + ' province' + (matches > 1 ? 's' : '') + ' matched';
            matchCount.style.color = matches === 0 ? '#cc0000' : '#1B5E8A';
        }}
        function clearSearch() {{
            document.getElementById('province-search').value = '';
            highlightProvince('');
        }}
    </script>
</body>
</html>""")
dashboard.close()
print("dashboard saved successfully")

In [ ]:
master_full['rank'] = range(1, len(master_full) + 1)
print(master_full[['ADM2_EN', 'overlap_score', 'rank']].head(5))

In [ ]:
fig_map = go.Figure(go.Choroplethmapbox(
    geojson=geojson_v2,
    locations=master_full['ADM2_PCODE'],
    z=master_full['overlap_score'],
    featureidkey='properties.ADM2_PCODE',
    colorscale='RdYlGn_r',
    zmin=master_full['overlap_score'].min(),
    zmax=master_full['overlap_score'].max(),
    marker_opacity=0.7,
    marker_line_width=0.5,
    marker_line_color='white',
    text=master_full['display_name'],
    customdata=master_full[['pm25','elderly','children_u5','rural_pop_perc','hospital_access_count','rank']].values,
    hovertemplate=(
        '<span style="font-size:15px"><b>%{text}</b></span><br>'
        '<span style="color:#888">Rank %{customdata[5]:.0f} of 88</span><br>'
        '<br>'
        '<b>Risk overlap score: %{z}</b><br>'
        '<br>'
        'PM2.5 (satellite 2022): %{customdata[0]} ug/m3<br>'
        'Elderly population (60+): %{customdata[1]:,}<br>'
        'Children under 5: %{customdata[2]:,}<br>'
        'Rural population: %{customdata[3]}%<br>'
        'Est. pop within 30min of hospital: %{customdata[4]:,}'
        '<extra></extra>'
    ),
    colorbar=dict(
        title=dict(
            text='Health risk<br>overlap score',
            font=dict(size=12),
            side='top'
        ),
        thickness=20,
        len=0.75,
        x=0.98,
        tickfont=dict(size=12),
        tickvals=[
            master_full['overlap_score'].min(),
            master_full['overlap_score'].quantile(0.25),
            master_full['overlap_score'].median(),
            master_full['overlap_score'].quantile(0.75),
            master_full['overlap_score'].max()
        ],
        ticktext=[
            str(round(master_full['overlap_score'].min(), 2)),
            str(round(master_full['overlap_score'].quantile(0.25), 2)),
            str(round(master_full['overlap_score'].median(), 2)),
            str(round(master_full['overlap_score'].quantile(0.75), 2)),
            str(round(master_full['overlap_score'].max(), 2))
        ],
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(0,0,0,0.2)',
        borderwidth=1
    )
))

fig_map.update_layout(
    mapbox=dict(
        accesstoken=mapbox_token,
        style='mapbox://styles/mapbox/light-v11',
        center=dict(lat=12.5, lon=122.5),
        zoom=4.5
    ),
    margin=dict(l=0, r=0, t=80, b=0),
    height=650,
    title=dict(
        text='Philippines Climate-Health Risk Index<br><sup>Air Pollution · Vulnerability · Healthcare Access Gap · NASA satellite PM2.5 2022</sup>',
        x=0.5,
        xanchor='center',
        font=dict(size=16)
    ),
    annotations=[],
    hoverlabel=dict(
        bgcolor='white',
        bordercolor='#1B5E8A',
        font=dict(size=13, family='Arial'),
        align='left'
    )
)

fig_map.show()

In [ ]:
# read existing dashboard
with open('/drive/MyDrive/philippines_risk_project/philippines_health_risk_dashboard_final.html', 'r') as f:
    content = f.read()

# highlight script to inject
highlight_script = f"""
<script>
var mapDiv;
function initMapHover() {{
    mapDiv = document.querySelector('#tab-map .js-plotly-plot');
    if (!mapDiv) {{ setTimeout(initMapHover, 500); return; }}

    mapDiv.on('plotly_hover', function(data) {{
        var idx = data.points[0].pointIndex;
        var numProvinces = {len(master_full)};
        var lineWidths = Array(numProvinces).fill(0.5);
        var lineColors = Array(numProvinces).fill('white');
        lineWidths[idx] = 3;
        lineColors[idx] = '#FFD700';
        Plotly.restyle(mapDiv, {{
            'marker.line.width': [lineWidths],
            'marker.line.color': [lineColors]
        }});
    }});

    mapDiv.on('plotly_unhover', function() {{
        var numProvinces = {len(master_full)};
        Plotly.restyle(mapDiv, {{
            'marker.line.width': [Array(numProvinces).fill(0.5)],
            'marker.line.color': [Array(numProvinces).fill('white')]
        }});
    }});
}}
initMapHover();
</script>
"""

# inject before closing body tag
content = content.replace('</body>', highlight_script + '</body>')

# save back
with open('/drive/MyDrive/philippines_risk_project/philippines_health_risk_dashboard_final.html', 'w') as f:
    f.write(content)

print("highlight effect injected successfully")

In [ ]:
fig_map.show()

In [ ]:
# read existing dashboard
with open('/drive/MyDrive/philippines_risk_project/philippines_health_risk_dashboard_final.html', 'r') as f:
    content = f.read()

# inject hoverlabel styling
hoverlabel_script = """
<script>
function applyHoverLabel() {
    var mapDiv = document.querySelector('#tab-map .js-plotly-plot');
    if (!mapDiv) { setTimeout(applyHoverLabel, 500); return; }
    Plotly.relayout(mapDiv, {
        'hoverlabel.bgcolor': 'white',
        'hoverlabel.bordercolor': '#1B5E8A',
        'hoverlabel.font.size': 13,
        'hoverlabel.font.family': 'Arial',
        'hoverlabel.align': 'left'
    });
}
applyHoverLabel();
</script>
"""

content = content.replace('</body>', hoverlabel_script + '</body>')

with open('/drive/MyDrive/philippines_risk_project/philippines_health_risk_dashboard_final.html', 'w') as f:
    f.write(content)

print("hoverlabel styling injected successfully")

In [ ]:
master_full['country'] = 'Philippines'
master_full.to_csv('/drive/MyDrive/philippines_risk_project/master_risk_final_v2.csv', index=False)
print("country column added successfully")

In [ ]:
fig_map.update_layout(
    mapbox=dict(
        accesstoken=mapbox_token,
        style='mapbox://styles/mapbox/satellite-streets-v12',
        center=dict(lat=15, lon=118),
        zoom=4
    ),
    margin=dict(l=0, r=0, t=80, b=0),
    height=650,
    title=dict(
        text='Asia Climate-Health Risk Index<br><sup>Air Pollution · Vulnerability · Healthcare Access Gap · Currently showing: Philippines (88 provinces)</sup>',
        x=0.5,
        xanchor='center',
        font=dict(size=16)
    )
)

fig_map.show()

In [ ]:
# rebuild dashboard + inject all features in one shot
map_html = fig_map.to_html(full_html=False, include_plotlyjs=True, config={'scrollZoom': True, 'displayModeBar': False})
bar_html = fig_bar.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': False})
province_json = df_sorted[['display_name', 'overlap_score']].to_json(orient='records')

dashboard = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Asia Climate-Health Risk Index</title>
    <style>
        * {{ box-sizing: border-box; margin: 0; padding: 0; font-family: Arial, sans-serif; }}
        body {{ background: #f4f4f4; }}
        .header {{ background: #1B5E8A; color: white; padding: 20px 30px; }}
        .header h1 {{ font-size: 20px; font-weight: 600; margin-bottom: 6px; }}
        .header p {{ font-size: 13px; opacity: 0.85; }}
        .tabs {{ display: flex; background: #ffffff; border-bottom: 2px solid #1B5E8A; padding: 0 30px; }}
        .tab {{ padding: 12px 24px; cursor: pointer; font-size: 14px; color: #555; border-bottom: 3px solid transparent; margin-bottom: -2px; transition: all 0.2s; }}
        .tab.active {{ background: #1B5E8A; color: white; font-weight: 600; border-bottom: 3px solid #1B5E8A; }}
        .panel {{ display: none; padding: 20px 30px; }}
        .panel.active {{ display: block; }}
        .map-container {{ background: white; border-radius: 8px; overflow: hidden; box-shadow: 0 1px 4px rgba(0,0,0,0.1); position: relative; }}
        .zoom-buttons {{ position: absolute; bottom: 30px; left: 20px; z-index: 999; display: flex; flex-direction: column; gap: 4px; }}
        .zoom-btn {{ width: 32px; height: 32px; background: white; border: 1px solid #ccc; border-radius: 4px; font-size: 18px; cursor: pointer; display: flex; align-items: center; justify-content: center; box-shadow: 0 1px 3px rgba(0,0,0,0.2); color: #333; }}
        .zoom-btn:hover {{ background: #f0f0f0; }}
        .bar-container {{ background: white; border-radius: 8px; overflow: hidden; box-shadow: 0 1px 4px rgba(0,0,0,0.1); }}
        .search-wrap {{ padding: 16px 20px 8px; background: white; border-radius: 8px 8px 0 0; display: flex; align-items: center; gap: 12px; border-bottom: 0.5px solid #eee; flex-wrap: wrap; }}
        .search-input {{ flex: 1; padding: 8px 14px; border: 1px solid #ccc; border-radius: 6px; font-size: 14px; outline: none; max-width: 360px; }}
        .search-input:focus {{ border-color: #1B5E8A; }}
        .search-label {{ font-size: 13px; color: #666; }}
        .match-count {{ font-size: 13px; color: #1B5E8A; font-weight: 600; min-width: 120px; }}
        .clear-btn {{ padding: 6px 12px; background: #f0f0f0; border: 1px solid #ccc; border-radius: 6px; font-size: 13px; cursor: pointer; color: #555; }}
        .clear-btn:hover {{ background: #e0e0e0; }}
        .country-filter {{ padding: 6px 12px; border: 1px solid #ccc; border-radius: 6px; font-size: 13px; color: #444; background: white; cursor: pointer; }}
        .meta {{ font-size: 12px; color: #888; margin-top: 10px; padding: 0 4px; }}
        .methodology {{ background: white; border-radius: 8px; padding: 16px 20px; margin-top: 16px; box-shadow: 0 1px 4px rgba(0,0,0,0.1); font-size: 13px; color: #444; line-height: 1.7; }}
        .methodology h3 {{ font-size: 14px; color: #1B5E8A; margin-bottom: 10px; font-weight: 600; }}
        .indicator-row {{ display: flex; gap: 12px; flex-wrap: wrap; margin-top: 10px; }}
        .indicator {{ background: #f0f5fa; border-radius: 6px; padding: 10px 14px; flex: 1; min-width: 180px; }}
        .indicator-name {{ font-weight: 600; color: #1B5E8A; font-size: 13px; margin-bottom: 4px; }}
        .indicator-desc {{ font-size: 12px; color: #666; }}
        .indicator-weight {{ font-size: 12px; font-weight: 600; color: #444; margin-top: 4px; }}
        .footer {{ background: #1B5E8A; color: rgba(255,255,255,0.7); font-size: 12px; padding: 12px 30px; margin-top: 30px; }}
    </style>
</head>
<body>
    <div class="header">
        <h1>Asia Climate-Health Risk Index</h1>
        <p>Air Pollution · Population Vulnerability · Healthcare Access Gap · Currently showing: Philippines (88 provinces)</p>
    </div>
    <div class="tabs">
        <div class="tab active" onclick="showTab('map', this)">Risk Map</div>
        <div class="tab" onclick="showTab('bar', this)">Province Rankings</div>
        <div class="tab" onclick="showTab('about', this)">About this index</div>
    </div>
    <div id="tab-map" class="panel active">
        <div class="map-container">
            {map_html}
            <div class="zoom-buttons">
                <button class="zoom-btn" onclick="zoomIn()">+</button>
                <button class="zoom-btn" onclick="zoomOut()">−</button>
            </div>
        </div>
        <p class="meta">Hover over any province to see detailed indicators. Scroll to zoom, click and drag to pan. Red = highest compounded risk, Green = lowest.</p>
    </div>
    <div id="tab-bar" class="panel">
        <div class="search-wrap">
            <span class="search-label">Filter by country:</span>
            <select class="country-filter" id="country-filter" onchange="filterCountry(this.value)">
                <option value="all">All countries</option>
                <option value="Philippines">Philippines</option>
            </select>
            <span class="search-label">Search province:</span>
            <input class="search-input" id="province-search" placeholder="Type a province name e.g. Cebu, Benguet..." oninput="highlightProvince(this.value)">
            <span class="match-count" id="match-count"></span>
            <button class="clear-btn" onclick="clearSearch()">Clear</button>
        </div>
        <div class="bar-container" id="bar-chart-container">
            {bar_html}
        </div>
        <p class="meta">All provinces ranked by health risk overlap score. Filter by country, then search to highlight. Hover over any bar for full indicator breakdown.</p>
    </div>
    <div id="tab-about" class="panel">
        <div class="methodology">
            <h3>How the health risk overlap score is calculated</h3>
            <p>The overlap score is a composite index ranging from 0 to 1 that combines three dimensions of health risk. A higher score means a province has greater compounded risk across all three dimensions simultaneously.</p>
            <div class="indicator-row">
                <div class="indicator">
                    <div class="indicator-name">Air pollution — PM2.5</div>
                    <div class="indicator-desc">Annual mean fine particulate matter (μg/m³) from NASA satellite data (MODIS, MISR, VIIRS, 2022). Strongest direct evidence linking air pollution to cardiovascular mortality.</div>
                    <div class="indicator-weight">Weight: 40%</div>
                </div>
                <div class="indicator">
                    <div class="indicator-name">Population vulnerability</div>
                    <div class="indicator-desc">Elderly population (60+) and children under 5 — the two age groups most vulnerable to air pollution and least able to cope with health shocks. Source: HDX / PSA.</div>
                    <div class="indicator-weight">Elderly: 20% · Children: 10%</div>
                </div>
                <div class="indicator">
                    <div class="indicator-name">Healthcare access gap</div>
                    <div class="indicator-desc">Estimated population within 30 minutes of a hospital, inverted so lower access = higher risk. Source: HDX / OCHA.</div>
                    <div class="indicator-weight">Weight: 30%</div>
                </div>
            </div>
            <p style="margin-top:14px"><b>Formula:</b> Overlap score = (PM2.5 × 0.40) + (Elderly × 0.20) + (Children × 0.10) + (Access gap × 0.30)</p>
            <p style="margin-top:8px"><b>Data sources:</b> NASA SEDAC V5.GL.04 (2022) · HDX PHL Risk Assessment · PSA · OCHA</p>
            <p style="margin-top:8px"><b>Limitations:</b> PM2.5 reflects satellite estimates not ground measurements. Hospital access reflects modelled travel times not actual utilisation rates.</p>
        </div>
    </div>
    <div class="footer">
        PM2.5 data: NASA SEDAC 2022 · Vulnerability data: PSA 2020 · Healthcare access: OCHA · Built with Python, Plotly and Mapbox
    </div>
    <script>
        function showTab(name, el) {{
            document.querySelectorAll('.panel').forEach(p => p.classList.remove('active'));
            document.querySelectorAll('.tab').forEach(t => t.classList.remove('active'));
            document.getElementById('tab-' + name).classList.add('active');
            el.classList.add('active');
        }}
        var currentZoom = 4;
        function zoomIn() {{
            currentZoom = Math.min(currentZoom + 0.5, 12);
            var mapDiv = document.querySelector('.js-plotly-plot');
            if (mapDiv) Plotly.relayout(mapDiv, {{'mapbox.zoom': currentZoom}});
        }}
        function zoomOut() {{
            currentZoom = Math.max(currentZoom - 0.5, 1);
            var mapDiv = document.querySelector('.js-plotly-plot');
            if (mapDiv) Plotly.relayout(mapDiv, {{'mapbox.zoom': currentZoom}});
        }}
        var provinceData = {province_json};
        function filterCountry(country) {{
            clearSearch();
        }}
        function highlightProvince(query) {{
            var barDiv = document.getElementById('bar-chart-container').querySelector('.js-plotly-plot');
            if (!barDiv) return;
            var matchCount = document.getElementById('match-count');
            if (!query || query.length < 2) {{
                Plotly.restyle(barDiv, {{'marker.opacity': [provinceData.map(function() {{ return 1; }})], 'marker.line.width': [provinceData.map(function() {{ return 0; }})]}});
                matchCount.textContent = '';
                return;
            }}
            var q = query.toLowerCase();
            var matches = 0;
            var opacities = provinceData.map(function(d) {{ var m = d.display_name.toLowerCase().indexOf(q) !== -1; if (m) matches++; return m ? 1 : 0.15; }});
            var lineWidths = provinceData.map(function(d) {{ return d.display_name.toLowerCase().indexOf(q) !== -1 ? 2 : 0; }});
            Plotly.restyle(barDiv, {{'marker.opacity': [opacities], 'marker.line.width': [lineWidths], 'marker.line.color': ['#1B5E8A']}});
            matchCount.textContent = matches === 0 ? 'No match found' : matches + ' province' + (matches > 1 ? 's' : '') + ' matched';
            matchCount.style.color = matches === 0 ? '#cc0000' : '#1B5E8A';
        }}
        function clearSearch() {{
            document.getElementById('province-search').value = '';
            highlightProvince('');
        }}
        var mapDiv2;
        function initMapHover() {{
            mapDiv2 = document.querySelector('#tab-map .js-plotly-plot');
            if (!mapDiv2) {{ setTimeout(initMapHover, 500); return; }}
            mapDiv2.on('plotly_hover', function(data) {{
                var idx = data.points[0].pointIndex;
                var n = {len(master_full)};
                var lw = Array(n).fill(0.5);
                var lc = Array(n).fill('white');
                lw[idx] = 3; lc[idx] = '#FFD700';
                Plotly.restyle(mapDiv2, {{'marker.line.width': [lw], 'marker.line.color': [lc]}});
            }});
            mapDiv2.on('plotly_unhover', function() {{
                var n = {len(master_full)};
                Plotly.restyle(mapDiv2, {{'marker.line.width': [Array(n).fill(0.5)], 'marker.line.color': [Array(n).fill('white')]}});
            }});
        }}
        function applyHoverLabel() {{
            var mapDiv2 = document.querySelector('#tab-map .js-plotly-plot');
            if (!mapDiv2) {{ setTimeout(applyHoverLabel, 500); return; }}
            Plotly.relayout(mapDiv2, {{
                'hoverlabel.bgcolor': 'white',
                'hoverlabel.bordercolor': '#1B5E8A',
                'hoverlabel.font.size': 13,
                'hoverlabel.font.family': 'Arial',
                'hoverlabel.align': 'left'
            }});
        }}
        initMapHover();
        applyHoverLabel();
    </script>
</body>
</html>"""

with open('/drive/MyDrive/philippines_risk_project/philippines_health_risk_dashboard_final.html', 'w') as f:
    f.write(dashboard)

print("dashboard saved successfully — all features included")

In [ ]:
# base path — change this one line to update everything
BASE = '/drive/MyDrive/asia_geospatial_project/'

# then load files like this
master_risk_final = pd.read_csv(BASE + 'master_risk_final_v2.csv')
vuln = pd.read_csv(BASE + 'PHL_ADM2_vulnerability.csv')
acc = pd.read_csv(BASE + 'PHL_ADM2_access.csv')
df = pd.read_csv(BASE + 'climate_air_quality.csv')

In [ ]:
from google.colab import drive
drive.mount('/drive')

import pandas as pd
import geopandas as gpd
import json

BASE = '/drive/MyDrive/asia_geospatial_project/'

master_risk_final = pd.read_csv(BASE + 'master_risk_final_v2.csv')
vuln = pd.read_csv(BASE + 'PHL_ADM2_vulnerability.csv')
acc = pd.read_csv(BASE + 'PHL_ADM2_access.csv')
crosswalk_raw = pd.read_excel(BASE + 'phl_adminboundaries_tabulardata.xlsx',
                               sheet_name='ADM2', engine='openpyxl')
crosswalk = crosswalk_raw[['ADM2_EN', 'ADM2_PCODE']]

with open(BASE + 'adm2_simplified.geojson') as f:
    geojson_v2 = json.load(f)

print("all files reloaded successfully")
print("base path:", BASE)

In [ ]:
import shutil
import os

# get the current notebook path
notebook_path = '/content/drive/MyDrive/asia_geospatial_project/asia_health_risk_pipeline.ipynb'

# save current notebook to drive
from google.colab import _message
_message.blocking_request('save_notebook', request='', timeout_sec=10)

print("saved")

In [ ]:
from  google.colab import drive
drive.mount('/drive')

import pandas as pd
import geopandas as gpd
import json
import rasterio
from rasterstats import zonal_stats

BASE = '/drive/MyDrive/asia_geospatial_project/'

master_risk_final = pd.read_csv(BASE + 'master_risk_final_v2.csv')
crosswalk_raw = pd.read_excel(BASE + 'phl_adminboundaries_tabulardata.xlsx',
sheet_name='ADM2', engine='openpyxl')
crosswalk = crosswalk_raw[['ADM2_EN', 'ADM2_PCODE']]

print("reloaded successfully")
print("Philippines provinces", len(master_risk_final))


In [ ]:
import os

# list everything in MyDrive
print(os.listdir('/drive/MyDrive/'))



In [ ]:
BASE = '/drive/MyDrive/asia-health-risk-geodatabase/'

# check what's inside
import os
print(os.listdir(BASE))

In [ ]:
BASE = '/drive/MyDrive/asia-health-risk-geodatabase/'
import pandas as pd
import geopandas as gpd
import json
import rasterio
import zonal_stats

master_risk_final = pd.read_csv(BASE + 'master_risk_final_v2.csv')
crosswalk_raw = pd.read_excel(BASE + 'phl_adminboundaries_tabulardata.xlsx',
                              sheet_name='ADM2', engine= 'openpyxl')
crosswalk= crosswalk_raw[['ADM2_EN', 'ADM2_PCODE']]

print("reloaded sucessfully")
print("Philippines provinces:", len(master_risk_final))

In [ ]:
!pip install rasterstats --quiet

In [ ]:
BASE = '/drive/MyDrive/asia-health-risk-geodatabase/'

import pandas as pd
import geopandas as gpd
import json
import rasterio
from rasterstats import zonal_stats

master_risk_final = pd.read_csv(BASE + 'master_risk_final_v2.csv')
crosswalk_raw = pd.read_excel(BASE + 'phl_adminboundaries_tabulardata.xlsx',
                               sheet_name='ADM2', engine='openpyxl')
crosswalk = crosswalk_raw[['ADM2_EN', 'ADM2_PCODE']]

print("reloaded successfully")
print("Philippines provinces:", len(master_risk_final))

In [ ]:
import os
print(os.listdir('/drive/MyDrive/'))

In [ ]:
from google.colab import drive
drive.mount('/drive', force_remount=True)

In [ ]:
import os
print(os.listdir('/drive/MyDrive/'))

In [ ]:
BASE = '/drive/MyDrive/asia-health-risk-geodatabase/'

import os
print(os.listdir(BASE))

In [ ]:
import pandas as pd
import geopandas as gpd
import json
import rasterio
from rasterstats import zonal_stats

master_risk_final = pd.read_csv(BASE + 'master_risk_final_v2.csv')
crosswalk_raw = pd.read_excel(BASE + 'phl_adminboundaries_tabulardata.xlsx',
                               sheet_name='ADM2', engine='openpyxl')
crosswalk = crosswalk_raw[['ADM2_EN', 'ADM2_PCODE']]

print("reloaded successfully")
print("Philippines provinces:", len(master_risk_final))

In [ ]:
taiwan = gpd.read_file(BASE + 'gadm41_TWN_1.json')
print("Taiwan shape:", taiwan.shape)
print("columns:", taiwan.columns.tolist())
print(taiwan[['NAME_1', 'GID_1']].head(10))

In [ ]:
taiwan2 = gpd.read_file(BASE +  'gadm41_TWN_2.json')
print("Taiwan shape:", taiwan2.shape)
print(taiwan2[['NAME_1', 'NAME_2', 'GID_2']].head(10))


In [ ]:
from google.colab import drive
drive.mount('/drive', force_remount=True)

import pandas as pd
import geopandas as gpd
import json
import rasterio
from rasterstats import zonal_stats

BASE = '/drive/MyDrive/asia-health-risk-geodatabase/'

master_risk_final = pd.read_csv(BASE + 'master_risk_final_v2.csv')

print("reloaded successfully")

In [ ]:
taiwan2 = gpd.read_file(BASE + 'gadm41_TWN_2.json')
print("Taiwan shape:", taiwan2.shape)
print(taiwan2[['NAME_1', 'NAME_2', 'GID_2']].head)

In [ ]:
tif_path = BASE + 'sdei-global-annual-gwr-pm2-5-modis-misr-seawifs-viirs-aod-v5-gl-04-2022-geotiff.tif'
nodata_value = -3.3999999521443642e+38

stats = zonal_stats(
    taiwan2,
    tif_path,
    stats=['mean'],
    nodata=nodata_value
)

taiwan2['pm25'] = [s['mean'] for s in stats]
taiwan2['pm25'] = taiwan2['pm25'].round(2)

print(taiwan2[['NAME_2', 'pm25']].sort_values('pm25', ascending=False))

In [ ]:
import requests

# Taiwan open statistics API - population by age and county
url = "https://ws.dgbas.gov.tw/001/Upload/461/relfile/10488/214016/county_age.csv"
response = requests.get(url)
print("status:", response.status_code)
print("first 500 chars:", response.text[:500])

In [ ]:
response = requests.get(url, verify=False)
print("status:", response.status_code)
print("first 500 chars:", response.text[:500])

In [ ]:
print(master_risk_final[master_risk_final['ADM2_EN'].str.contains('Manila|Metropolitan', na=False)][['ADM2_EN', 'overlap_score', 'pm25']])

In [ ]:
import geopandas as gpd

provinces = gpd.read_file(BASE + 'philippines_risk_v2.geojson')
print(provinces[provinces['ADM2_EN'].str.contains('Manila|Metropolitan', na=False)][['ADM2_EN', 'overlap_score', 'pm25']])

In [ ]:
print(provinces.columns.tolist())

In [ ]:
# reload clean province boundaries
provinces_clean = gpd.read_file(BASE + 'adm2_simplified.geojson')
print("boundary columns:", provinces_clean.columns.tolist())
print("boundary shape:", provinces_clean.shape)

In [ ]:
import geopandas as gpd

# reload original shapefile directly
phl_shp = gpd.read_file('/drive/MyDrive/asia-health-risk-geodatabase/adm2_simplified.geojson')

# keep only the essential columns
phl_clean = phl_shp[['ADM2_PCODE', 'ADM2_EN', 'geometry']].copy()

print("clean boundary shape:", phl_clean.shape)
print("columns:", phl_clean.columns.tolist())
print(phl_clean[phl_clean['ADM2_EN'].str.contains('Manila|Metropolitan', na=False)][['ADM2_EN', 'ADM2_PCODE']])

In [ ]:
# merge clean boundaries with master risk data
phl_final = phl_clean.merge(
    master_risk_final[['ADM2_PCODE', 'ADM2_EN', 'pm25', 'elderly',
                        'children_u5', 'rural_pop_perc', 'hospital_access_count',
                        'overlap_score', 'rank', 'country']],
    on='ADM2_PCODE',
    how='left'
)

# check Manila is there with data
print(phl_final[phl_final['ADM2_EN_x'].str.contains('Manila|Metropolitan', na=False)][['ADM2_EN_x', 'overlap_score', 'pm25']])
print("\ntotal provinces:", len(phl_final))
print("provinces with overlap score:", phl_final['overlap_score'].notna().sum())

In [ ]:
# save clean geojson
phl_final.to_file(BASE + 'philippines_risk_v3.geojson', driver='GeoJSON')
print("saved successfully")

In [ ]:
print(master_risk_final.columns.tolist())

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# load clean geojson
with open(BASE + 'philippines_risk_v3.geojson') as f:
    geojson_clean = json.load(f)

# sort for bar chart
df_sorted = master_risk_final.sort_values('overlap_score', ascending=True)

mapbox_token = "pk.eyJ1IjoieW5uZ3V5ZW4iLCJhIjoiY21teDR3NzN2MTE0czJ1b2pjcTJ6d3Y2aCJ9.ATgZczFgyaW9dcHWCBMdzg"

fig_map = go.Figure(go.Choroplethmapbox(
    geojson=geojson_clean,
    locations=master_risk_final['ADM2_PCODE'],
    z=master_risk_final['overlap_score'],
    featureidkey='properties.ADM2_PCODE',
    colorscale='RdYlGn_r',
    zmin=master_risk_final['overlap_score'].min(),
    zmax=master_risk_final['overlap_score'].max(),
    marker_opacity=0.7,
    marker_line_width=0.5,
    marker_line_color='white',
    text=master_risk_final['display_name'],
    customdata=master_risk_final[['pm25','elderly','children_u5','rural_pop_perc','hospital_access_count','rank']].values,
    hovertemplate=(
        '<span style="font-size:15px"><b>%{text}</b></span><br>'
        '<span style="color:#888">Rank %{customdata[5]:.0f} of 88 · Philippines</span><br>'
        '<br>'
        '<b>Risk overlap score: %{z}</b><br>'
        '<br>'
        'PM2.5 (satellite 2022): %{customdata[0]} ug/m3<br>'
        'Elderly population (60+): %{customdata[1]:,}<br>'
        'Children under 5: %{customdata[2]:,}<br>'
        'Rural population: %{customdata[3]}%<br>'
        'Est. pop within 30min of hospital: %{customdata[4]:,}'
        '<extra></extra>'
    ),
    colorbar=dict(
        title=dict(
            text='Health risk<br>overlap score',
            font=dict(size=12),
            side='top'
        ),
        thickness=20,
        len=0.75,
        x=0.98,
        tickfont=dict(size

In [ ]:
fig_map.show()

In [ ]:
# dashboard reload cell

map_html = fig_map.to_html(full_html=False, include_plotlyjs=True, config={'scrollZoom': True, 'displayModeBar': False})
bar_html = fig_bar.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': False})
province_json = df_sorted[['display_name', 'overlap_score']].to_json(orient='records')

dashboard = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Asia Climate-Health Risk Index</title>
    <style>
        * {{ box-sizing: border-box; margin: 0; padding: 0; font-family: Arial, sans-serif; }}
        body {{ background: #f4f4f4; }}
        .header {{ background: #1B5E8A; color: white; padding: 20px 30px; }}
        .header h1 {{ font-size: 20px; font-weight: 600; margin-bottom: 6px; }}
        .header p {{ font-size: 13px; opacity: 0.85; }}
        .tabs {{ display: flex; background: #ffffff; border-bottom: 2px solid #1B5E8A; padding: 0 30px; }}
        .tab {{ padding: 12px 24px; cursor: pointer; font-size: 14px; color: #555; border-bottom: 3px solid transparent; margin-bottom: -2px; transition: all 0.2s; }}
        .tab.active {{ background: #1B5E8A; color: white; font-weight: 600; border-bottom: 3px solid #1B5E8A; }}
        .panel {{ display: none; padding: 20px 30px; }}
        .panel.active {{ display: block; }}
        .map-container {{ background: white; border-radius: 8px; overflow: hidden; box-shadow: 0 1px 4px rgba(0,0,0,0.1); position: relative; }}
        .zoom-buttons {{ position: absolute; bottom: 30px; left: 20px; z-index: 999; display: flex; flex-direction: column; gap: 4px; }}
        .zoom-btn {{ width: 32px; height: 32px; background: white; border: 1px solid #ccc; border-radius: 4px; font-size: 18px; cursor: pointer; display: flex; align-items: center; justify-content: center; box-shadow: 0 1px 3px rgba(0,0,0,0.2); color: #333; }}
        .zoom-btn:hover {{ background: #f0f0f0; }}
        .bar-container {{ background: white; border-radius: 8px; overflow: hidden; box-shadow: 0 1px 4px rgba(0,0,0,0.1); }}
        .search-wrap {{ padding: 16px 20px 8px; background: white; border-radius: 8px 8px 0 0; display: flex; align-items: center; gap: 12px; border-bottom: 0.5px solid #eee; flex-wrap: wrap; }}
        .search-input {{ flex: 1; padding: 8px 14px; border: 1px solid #ccc; border-radius: 6px; font-size: 14px; outline: none; max-width: 360px; }}
        .search-input:focus {{ border-color: #1B5E8A; }}
        .search-label {{ font-size: 13px; color: #666; }}
        .match-count {{ font-size: 13px; color: #1B5E8A; font-weight: 600; min-width: 120px; }}
        .clear-btn {{ padding: 6px 12px; background: #f0f0f0; border: 1px solid #ccc; border-radius: 6px; font-size: 13px; cursor: pointer; color: #555; }}
        .clear-btn:hover {{ background: #e0e0e0; }}
        .country-filter {{ padding: 6px 12px; border: 1px solid #ccc; border-radius: 6px; font-size: 13px; color: #444; background: white; cursor: pointer; }}
        .meta {{ font-size: 12px; color: #888; margin-top: 10px; padding: 0 4px; }}
        .methodology {{ background: white; border-radius: 8px; padding: 16px 20px; margin-top: 16px; box-shadow: 0 1px 4px rgba(0,0,0,0.1); font-size: 13px; color: #444; line-height: 1.7; }}
        .methodology h3 {{ font-size: 14px; color: #1B5E8A; margin-bottom: 10px; font-weight: 600; }}
        .indicator-row {{ display: flex; gap: 12px; flex-wrap: wrap; margin-top: 10px; }}
        .indicator {{ background: #f0f5fa; border-radius: 6px; padding: 10px 14px; flex: 1; min-width: 180px; }}
        .indicator-name {{ font-weight: 600; color: #1B5E8A; font-size: 13px; margin-bottom: 4px; }}
        .indicator-desc {{ font-size: 12px; color: #666; }}
        .indicator-weight {{ font-size: 12px; font-weight: 600; color: #444; margin-top: 4px; }}
        .footer {{ background: #1B5E8A; color: rgba(255,255,255,0.7); font-size: 12px; padding: 12px 30px; margin-top: 30px; }}
    </style>
</head>
<body>
    <div class="header">
        <h1>Asia Climate-Health Risk Index</h1>
        <p>Air Pollution · Population Vulnerability · Healthcare Access Gap · Currently showing: Philippines (88 provinces)</p>
    </div>
    <div class="tabs">
        <div class="tab active" onclick="showTab('map', this)">Risk Map</div>
        <div class="tab" onclick="showTab('bar', this)">Province Rankings</div>
        <div class="tab" onclick="showTab('about', this)">About this index</div>
    </div>
    <div id="tab-map" class="panel active">
        <div class="map-container">
            {map_html}
            <div class="zoom-buttons">
                <button class="zoom-btn" onclick="zoomIn()">+</button>
                <button class="zoom-btn" onclick="zoomOut()">−</button>
            </div>
        </div>
        <p class="meta">Hover over any province to see detailed indicators. Scroll to zoom, click and drag to pan. Red = highest risk, Green = lowest.</p>
    </div>
    <div id="tab-bar" class="panel">
        <div class="search-wrap">
            <span class="search-label">Filter by country:</span>
            <select class="country-filter" id="country-filter" onchange="filterCountry(this.value)">
                <option value="all">All countries</option>
                <option value="Philippines">Philippines</option>
            </select>
            <span class="search-label">Search province:</span>
            <input class="search-input" id="province-search" placeholder="Type a province name e.g. Cebu, Benguet..." oninput="highlightProvince(this.value)">
            <span class="match-count" id="match-count"></span>
            <button class="clear-btn" onclick="clearSearch()">Clear</button>
        </div>
        <div class="bar-container" id="bar-chart-container">
            {bar_html}
        </div>
        <p class="meta">All 88 provinces ranked by health risk overlap score. Search to highlight. Hover for full indicator breakdown.</p>
    </div>
    <div id="tab-about" class="panel">
        <div class="methodology">
            <h3>How the health risk overlap score is calculated</h3>
            <p>The overlap score is a composite index ranging from 0 to 1 combining three dimensions of health risk. A higher score means greater compounded risk across all three dimensions simultaneously.</p>
            <div class="indicator-row">
                <div class="indicator">
                    <div class="indicator-name">Air pollution — PM2.5</div>
                    <div class="indicator-desc">Annual mean fine particulate matter (μg/m³) from NASA satellite data (MODIS, MISR, VIIRS, 2022). Strongest direct evidence linking air pollution to cardiovascular mortality.</div>
                    <div class="indicator-weight">Weight: 40%</div>
                </div>
                <div class="indicator">
                    <div class="indicator-name">Population vulnerability</div>
                    <div class="indicator-desc">Elderly population (60+) and children under 5 — most vulnerable to air pollution and least able to cope with health shocks. Source: HDX / PSA.</div>
                    <div class="indicator-weight">Elderly: 20% · Children: 10%</div>
                </div>
                <div class="indicator">
                    <div class="indicator-name">Healthcare access gap</div>
                    <div class="indicator-desc">Estimated population within 30 minutes of a hospital, inverted so lower access = higher risk. Source: HDX / OCHA.</div>
                    <div class="indicator-weight">Weight: 30%</div>
                </div>
            </div>
            <p style="margin-top:14px"><b>Formula:</b> Overlap score = (PM2.5 × 0.40) + (Elderly × 0.20) + (Children × 0.10) + (Access gap × 0.30)</p>
            <p style="margin-top:8px"><b>Data sources:</b> NASA SEDAC V5.GL.04 (2022) · HDX PHL Risk Assessment · PSA · OCHA</p>
            <p style="margin-top:8px"><b>Limitations:</b> PM2.5 reflects satellite estimates not ground measurements. Hospital access reflects modelled travel times not actual utilisation rates. AI-assisted Python development.</p>
        </div>
    </div>
    <div class="footer">
        PM2.5 data: NASA SEDAC 2022 · Vulnerability data: PSA 2020 · Healthcare access: OCHA · Built with Python, Plotly and Mapbox
    </div>
    <script>
        function showTab(name, el) {{
            document.querySelectorAll('.panel').forEach(p => p.classList.remove('active'));
            document.querySelectorAll('.tab').forEach(t => t.classList.remove('active'));
            document.getElementById('tab-' + name).classList.add('active');
            el.classList.add('active');
        }}
        var currentZoom = 4;
        function zoomIn() {{
            currentZoom = Math.min(currentZoom + 0.5, 12);
            var mapDiv = document.querySelector('.js-plotly-plot');
            if (mapDiv) Plotly.relayout(mapDiv, {{'mapbox.zoom': currentZoom}});
        }}
        function zoomOut() {{
            currentZoom = Math.max(currentZoom - 0.5, 1);
            var mapDiv = document.querySelector('.js-plotly-plot');
            if (mapDiv) Plotly.relayout(mapDiv, {{'mapbox.zoom': currentZoom}});
        }}
        var provinceData = {province_json};
        function filterCountry(country) {{ clearSearch(); }}
        function highlightProvince(query) {{
            var barDiv = document.getElementById('bar-chart-container').querySelector('.js-plotly-plot');
            if (!barDiv) return;
            var matchCount = document.getElementById('match-count');
            if (!query || query.length < 2) {{
                Plotly.restyle(barDiv, {{'marker.opacity': [provinceData.map(function() {{ return 1; }})], 'marker.line.width': [provinceData.map(function() {{ return 0; }})]}});
                matchCount.textContent = '';
                return;
            }}
            var q = query.toLowerCase();
            var matches = 0;
            var opacities = provinceData.map(function(d) {{ var m = d.display_name.toLowerCase().indexOf(q) !== -1; if (m) matches++; return m ? 1 : 0.15; }});
            var lineWidths = provinceData.map(function(d) {{ return d.display_name.toLowerCase().indexOf(q) !== -1 ? 2 : 0; }});
            Plotly.restyle(barDiv, {{'marker.opacity': [opacities], 'marker.line.width': [lineWidths], 'marker.line.color': ['#1B5E8A']}});
            matchCount.textContent = matches === 0 ? 'No match found' : matches + ' province' + (matches > 1 ? 's' : '') + ' matched';
            matchCount.style.color = matches === 0 ? '#cc0000' : '#1B5E8A';
        }}
        function clearSearch() {{
            document.getElementById('province-search').value = '';
            highlightProvince('');
        }}
        var mapDiv2;
        function initMapHover() {{
            mapDiv2 = document.querySelector('#tab-map .js-plotly-plot');
            if (!mapDiv2) {{ setTimeout(initMapHover, 500); return; }}
            mapDiv2.on('plotly_hover', function(data) {{
                var idx = data.points[0].pointIndex;
                var n = {len(master_risk_final)};
                var lw = Array(n).fill(0.5);
                var lc = Array(n).fill('white');
                lw[idx] = 3; lc[idx] = '#FFD700';
                Plotly.restyle(mapDiv2, {{'marker.line.width': [lw], 'marker.line.color': [lc]}});
            }});
            mapDiv2.on('plotly_unhover', function() {{
                var n = {len(master_risk_final)};
                Plotly.restyle(mapDiv2, {{'marker.line.width': [Array(n).fill(0.5)], 'marker.line.color': [Array(n).fill('white')]}});
            }});
        }}
        function applyHoverLabel() {{
            var mapDiv2 = document.querySelector('#tab-map .js-plotly-plot');
            if (!mapDiv2) {{ setTimeout(applyHoverLabel, 500); return; }}
            Plotly.relayout(mapDiv2, {{
                'hoverlabel.bgcolor': 'white',
                'hoverlabel.bordercolor': '#1B5E8A',
                'hoverlabel.font.size': 13,
                'hoverlabel.font.family': 'Arial',
                'hoverlabel.align': 'left'
            }});
        }}
        initMapHover();
        applyHoverLabel();
    </script>
</body>
</html>"""

with open(BASE + 'philippines_health_risk_dashboard_final.html', 'w') as f:
    f.write(dashboard)

print("dashboard saved successfully — all features included")

In [ ]:
print(df_sorted[df_sorted['display_name'].str.contains('Manila|Quezon City|Caloocan|Pasig', na=False)]['display_name'].tolist())

In [ ]:
# taiwan begin
import geopandas as gpd
from rasterstats import zonal_stats

tif_path = BASE + 'sdei-global-annual-gwr-pm2-5-modis-misr-seawifs-viirs-aod-v5-gl-04-2022-geotiff.tif'
nodata_value = -3.3999999521443642e+38

taiwan2 = gpd.read_file(BASE + 'gadm41_TWN_2.json')

stats = zonal_stats(
    taiwan2,
    tif_path,
    stats=['mean'],
    nodata=nodata_value
)

taiwan2['pm25'] = [s['mean'] for s in stats]
taiwan2['pm25'] = taiwan2['pm25'].round(2)

print(taiwan2[['NAME_2', 'GID_2', 'pm25']].to_string())

In [ ]:
taiwan_demographics = pd.DataFrame({
    'NAME_2': ['Kinmen', 'Lienkiang', 'Kaohsiung', 'NewTaipei', 'Taichung',
               'Tainan', 'Taipei', 'Changhua', 'ChiayiCity', 'ChiayiCounty',
               'HsinchuCity', 'HsinchuCounty', 'Hualien', 'Keelung', 'Miaoli',
               'Nantou', 'Penghu', 'Pingtung', 'Taitung', 'Taoyuan',
               'Yilan', 'Yulin'],
    'elderly': [14800, 2100, 622000, 820000, 530000,
                378000, 448000, 198000, 56000, 98000,
                66000, 98000, 72000, 78000, 98000,
                88000, 18000, 168000, 72000, 440000,
                82000, 148000],
    'children_u5': [3200, 420, 98000, 148000, 118000,
                    68000, 52000, 32000, 7200, 12000,
                    18000, 22000, 8800, 10000, 12000,
                    9200, 2800, 22000, 8400, 128000,
                    12000, 18000],
    'rural_pop_perc': [35.2, 82.4, 12.8, 8.4, 15.2,
                       18.6, 2.1, 42.3, 8.9, 68.4,
                       6.2, 48.7, 72.3, 5.4, 58.2,
                       74.6, 45.8, 52.4, 68.9, 8.8,
                       62.4, 58.6]
})

print(taiwan_demographics)
print("\nshape:", taiwan_demographics.shape)

In [ ]:
taiwan_health = pd.read_excel(BASE + 'Statistics_of_Medical_Care_Institution_Utilization_2024.xlsx')
print(taiwan_health.columns.tolist())
print(taiwan_health.shape)
print(taiwan_health.head(5).to_string())

In [ ]:
xl = pd.ExcelFile(BASE + 'Statistics_of_Medical_Care_Institution_Utilization_2024.xlsx')
print(xl.sheet_names)

In [ ]:
table4 = pd.read_excel(BASE + 'Statistics_of_Medical_Care_Institution_Utilization_2024.xlsx',
                        sheet_name='04', header=None)
print(table4.to_string())

In [ ]:
# check table 20 which might have county breakdown
table20 = pd.read_excel(BASE + 'Statistics_of_Medical_Care_Institution_Utilization_2024.xlsx',
                        sheet_name='20', header=None)
print(table20.iloc[:10, :5].to_string())

In [ ]:
table20 = pd.read_excel(BASE + 'Statistics_of_Medical_Care_Institution_Utilization_2024.xlsx',
                        sheet_name='20', header=None)

# print more rows to see all counties
print(table20.iloc[7:35, :5].to_string())

In [ ]:
# extract county data
taiwan_hospitals = pd.DataFrame({
    'county_raw': table20.iloc[8:30, 0].values,
    'total_facilities': table20.iloc[8:30, 2].values,
    'total_hospitals': table20.iloc[8:30, 3].values
})

# extract English names from the raw column
taiwan_hospitals['county_en'] = taiwan_hospitals['county_raw'].str.extract(r'([A-Za-z\s]+(?:City|County))')
taiwan_hospitals['county_en'] = taiwan_hospitals['county_en'].str.strip()

print(taiwan_hospitals[['county_en', 'total_facilities', 'total_hospitals']])

In [ ]:
# extract English names by splitting on newline
taiwan_hospitals['county_en'] = taiwan_hospitals['county_raw'].str.split('\n').str[-1].str.strip()

print(taiwan_hospitals[['county_en', 'total_facilities', 'total_hospitals']])

In [ ]:
# manual mapping based on the order we saw in the table
taiwan_hospitals = pd.DataFrame({
    'NAME_2': ['NewTaipei', 'Taipei', 'Taoyuan', 'Taichung', 'Tainan',
               'Kaohsiung', 'Yilan', 'HsinchuCounty', 'Miaoli', 'Changhua',
               'Nantou', 'Yulin', 'ChiayiCounty', 'Pingtung', 'Taitung',
               'Hualien', 'Penghu', 'Keelung', 'HsinchuCity', 'ChiayiCity',
               'Kinmen', 'Lienkiang'],
    'total_hospitals': [52, 36, 34, 64, 34, 83, 9, 10, 14, 29,
                        10, 15, 4, 23, 7, 10, 3, 9, 9, 11, 1, 1],
    'total_facilities': [3576, 3960, 1857, 3767, 2076, 3149, 365, 457, 391, 1063,
                         411, 507, 273, 657, 163, 281, 87, 302, 460, 403, 58, 5]
})

print(taiwan_hospitals)
print("\nshape:", taiwan_hospitals.shape)

In [ ]:
# check sheet 01 which might have population
table01 = pd.read_excel(BASE + 'Statistics_of_Medical_Care_Institution_Utilization_2024.xlsx',
                        sheet_name='01', header=None)
print(table01.iloc[:10, :5].to_string())

In [ ]:
taiwan_population = pd.DataFrame({
    'NAME_2': ['Kinmen', 'Lienkiang', 'Kaohsiung', 'NewTaipei', 'Taichung',
               'Tainan', 'Taipei', 'Changhua', 'ChiayiCity', 'ChiayiCounty',
               'HsinchuCity', 'HsinchuCounty', 'Hualien', 'Keelung', 'Miaoli',
               'Nantou', 'Penghu', 'Pingtung', 'Taitung', 'Taoyuan',
               'Yilan', 'Yulin'],
    'total_pop': [140000, 13000, 2767000, 4030000, 2820000,
                  1880000, 2600000, 1270000, 265000, 510000,
                  450000, 560000, 330000, 370000, 550000,
                  490000, 107000, 840000, 340000, 2270000,
                  460000, 700000]
})

print(taiwan_population)

In [ ]:
# merge population into hospitals
taiwan_hospitals = taiwan_hospitals.merge(taiwan_population, on='NAME_2')

# calculate hospitals per 10,000 population
taiwan_hospitals['hospitals_per_10k'] = (
    taiwan_hospitals['total_hospitals'] / taiwan_hospitals['total_pop'] * 10000
).round(2)

print(taiwan_hospitals[['NAME_2', 'total_hospitals', 'total_pop', 'hospitals_per_10k']]
      .sort_values('hospitals_per_10k', ascending=False))

In [ ]:
# merge all Taiwan data together
taiwan_master = taiwan2[['NAME_2', 'GID_2', 'pm25', 'geometry']].copy()
taiwan_master = taiwan_master.merge(taiwan_demographics, on='NAME_2')
taiwan_master = taiwan_master.merge(taiwan_hospitals[['NAME_2', 'total_hospitals', 'hospitals_per_10k']], on='NAME_2')
taiwan_master = taiwan_master.merge(taiwan_population, on='NAME_2')

# add country column
taiwan_master['country'] = 'Taiwan'

print(taiwan_master.shape)
print(taiwan_master.columns.tolist())
print(taiwan_master[['NAME_2', 'pm25', 'elderly', 'children_u5', 'hospitals_per_10k']].head(5))

In [ ]:
# normalise each indicator to 0-1 scale
taiwan_master['pm25_norm'] = (taiwan_master['pm25'] - taiwan_master['pm25'].min()) / (taiwan_master['pm25'].max() - taiwan_master['pm25'].min())

taiwan_master['elderly_norm'] = (taiwan_master['elderly'] - taiwan_master['elderly'].min()) / (taiwan_master['elderly'].max() - taiwan_master['elderly'].min())

taiwan_master['children_norm'] = (taiwan_master['children_u5'] - taiwan_master['children_u5'].min()) / (taiwan_master['children_u5'].max() - taiwan_master['children_u5'].min())

# invert hospitals_per_10k so lower density = higher risk
taiwan_master['access_norm'] = 1 - (taiwan_master['hospitals_per_10k'] - taiwan_master['hospitals_per_10k'].min()) / (taiwan_master['hospitals_per_10k'].max() - taiwan_master['hospitals_per_10k'].min())

# weighted overlap score — same weights as Philippines
taiwan_master['overlap_score'] = (
    taiwan_master['pm25_norm'] * 0.40 +
    taiwan_master['elderly_norm'] * 0.20 +
    taiwan_master['children_norm'] * 0.10 +
    taiwan_master['access_norm'] * 0.30
).round(3)

taiwan_master = taiwan_master.sort_values('overlap_score', ascending=False)
taiwan_master['rank'] = range(1, len(taiwan_master) + 1)

print(taiwan_master[['NAME_2', 'pm25', 'elderly', 'hospitals_per_10k', 'overlap_score', 'rank']])

In [ ]:
# prepare Philippines data to match Taiwan structure
phl_combined = master_risk_final[['ADM2_PCODE', 'ADM2_EN', 'pm25', 'elderly',
                                   'children_u5', 'rural_pop_perc',
                                   'hospital_access_count', 'overlap_score',
                                   'rank', 'country', 'display_name']].copy()
phl_combined = phl_combined.rename(columns={'ADM2_PCODE': 'unit_id', 'ADM2_EN': 'unit_name'})

# prepare Taiwan data
twn_combined = taiwan_master[['GID_2', 'NAME_2', 'pm25', 'elderly',
                               'children_u5', 'rural_pop_perc',
                               'hospitals_per_10k', 'overlap_score',
                               'rank', 'country']].copy()
twn_combined = twn_combined.rename(columns={'GID_2': 'unit_id', 'NAME_2': 'unit_name'})
twn_combined['hospital_access_count'] = twn_combined['hospitals_per_10k']
twn_combined['display_name'] = twn_combined['unit_name']

# combine
asia_master = pd.concat([phl_combined, twn_combined], ignore_index=True)

print("Total units:", len(asia_master))
print("Philippines:", len(asia_master[asia_master['country'] == 'Philippines']))
print("Taiwan:", len(asia_master[asia_master['country'] == 'Taiwan']))

In [ ]:
# save Taiwan geojson
taiwan_master.to_file(BASE + 'taiwan_risk_v1.geojson', driver='GeoJSON')

# save combined master CSV
asia_master.to_csv(BASE + 'asia_master_v1.csv', index=False)

print("Taiwan GeoJSON saved")
print("Asia master CSV saved")
print(asia_master[['unit_name', 'country', 'overlap_score', 'rank']].head(10))

In [ ]:
import geopandas as gpd

# load both boundary files
phl_geo = gpd.read_file(BASE + 'philippines_risk_v3.geojson')[['ADM2_PCODE', 'ADM2_EN', 'geometry']].copy()
phl_geo = phl_geo.rename(columns={'ADM2_PCODE': 'unit_id', 'ADM2_EN': 'unit_name'})

twn_geo = taiwan_master[['GID_2', 'NAME_2', 'geometry']].copy()
twn_geo = twn_geo.rename(columns={'GID_2': 'unit_id', 'NAME_2': 'unit_name'})

# combine geometries
asia_geo = pd.concat([phl_geo, twn_geo], ignore_index=True)
asia_geo = gpd.GeoDataFrame(asia_geo, geometry='geometry', crs='EPSG:4326')

# merge with risk data
asia_geo = asia_geo.merge(
    asia_master[['unit_id', 'overlap_score', 'pm25', 'elderly',
                 'children_u5', 'rural_pop_perc', 'hospital_access_count',
                 'rank', 'country', 'display_name']],
    on='unit_id'
)

print("Combined GeoDataFrame shape:", asia_geo.shape)
print("Philippines units:", len(asia_geo[asia_geo['country'] == 'Philippines']))
print("Taiwan units:", len(asia_geo[asia_geo['country'] == 'Taiwan']))

# save combined geojson
asia_geo.to_file(BASE + 'asia_risk_v1.geojson', driver='GeoJSON')
print("Asia GeoJSON saved")

In [ ]:
phl_geo_check = gpd.read_file(BASE + 'philippines_risk_v3.geojson')
print(phl_geo_check.columns.tolist())

In [ ]:
# load philippines geojson - use ADM2_EN_x as the name column
phl_geo = gpd.read_file(BASE + 'philippines_risk_v3.geojson')[['ADM2_PCODE', 'ADM2_EN_x', 'geometry']].copy()
phl_geo = phl_geo.rename(columns={'ADM2_PCODE': 'unit_id', 'ADM2_EN_x': 'unit_name'})

twn_geo = taiwan_master[['GID_2', 'NAME_2', 'geometry']].copy()
twn_geo = twn_geo.rename(columns={'GID_2': 'unit_id', 'NAME_2': 'unit_name'})

# combine geometries
asia_geo = pd.concat([phl_geo, twn_geo], ignore_index=True)
asia_geo = gpd.GeoDataFrame(asia_geo, geometry='geometry', crs='EPSG:4326')

# merge with risk data
asia_geo = asia_geo.merge(
    asia_master[['unit_id', 'overlap_score', 'pm25', 'elderly',
                 'children_u5', 'rural_pop_perc', 'hospital_access_count',
                 'rank', 'country', 'display_name']],
    on='unit_id'
)

print("Combined GeoDataFrame shape:", asia_geo.shape)
print("Philippines units:", len(asia_geo[asia_geo['country'] == 'Philippines']))
print("Taiwan units:", len(asia_geo[asia_geo['country'] == 'Taiwan']))

# save combined geojson
asia_geo.to_file(BASE + 'asia_risk_v1.geojson', driver='GeoJSON')
print("Asia GeoJSON saved")

In [ ]:
import plotly.graph_objects as go
import json

mapbox_token = "pk.eyJ1IjoieW5uZ3V5ZW4iLCJhIjoiY21teDR3NzN2MTE0czJ1b2pjcTJ6d3Y2aCJ9.ATgZczFgyaW9dcHWCBMdzg"

# load combined geojson
with open(BASE + 'asia_risk_v1.geojson') as f:
    geojson_asia = json.load(f)

# sort for bar chart
df_sorted_asia = asia_master.sort_values('overlap_score', ascending=True)

# build combined map
fig_map = go.Figure(go.Choroplethmapbox(
    geojson=geojson_asia,
    locations=asia_master['unit_id'],
    z=asia_master['overlap_score'],
    featureidkey='properties.unit_id',
    colorscale='RdYlGn_r',
    zmin=asia_master['overlap_score'].min(),
    zmax=asia_master['overlap_score'].max(),
    marker_opacity=0.7,
    marker_line_width=0.5,
    marker_line_color='white',
    text=asia_master['display_name'],
    customdata=asia_master[['pm25','elderly','children_u5','rural_pop_perc','hospital_access_count','rank','country']].values,
    hovertemplate=(
        '<span style="font-size:15px"><b>%{text}</b></span><br>'
        '<span style="color:#888">%{customdata[6]} · Rank %{customdata[5]:.0f}</span><br>'
        '<br>'
        '<b>Risk overlap score: %{z}</b><br>'
        '<br>'
        'PM2.5 (satellite 2022): %{customdata[0]} ug/m3<br>'
        'Elderly population (60+): %{customdata[1]:,}<br>'
        'Children under 5: %{customdata[2]:,}<br>'
        'Rural population: %{customdata[3]}%<br>'
        'Healthcare access: %{customdata[4]:,}'
        '<extra></extra>'
    ),
    colorbar=dict(
        title=dict(
            text='Health risk<br>overlap score',
            font=dict(size=12),
            side='top'
        ),
        thickness=20,
        len=0.75,
        x=0.98,
        tickfont=dict(size=12),
        tickvals=[
            asia_master['overlap_score'].min(),
            asia_master['overlap_score'].quantile(0.25),
            asia_master['overlap_score'].median(),
            asia_master['overlap_score'].quantile(0.75),
            asia_master['overlap_score'].max()
        ],
        ticktext=[
            str(round(asia_master['overlap_score'].min(), 2)),
            str(round(asia_master['overlap_score'].quantile(0.25), 2)),
            str(round(asia_master['overlap_score'].median(), 2)),
            str(round(asia_master['overlap_score'].quantile(0.75), 2)),
            str(round(asia_master['overlap_score'].max(), 2))
        ],
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(0,0,0,0.2)',
        borderwidth=1
    )
))

fig_map.update_layout(
    mapbox=dict(
        accesstoken=mapbox_token,
        style='mapbox://styles/mapbox/satellite-streets-v12',
        center=dict(lat=18, lon=118),
        zoom=4
    ),
    margin=dict(l=0, r=0, t=80, b=0),
    height=650,
    title=dict(
        text='Asia Climate-Health Risk Index<br><sup>Air Pollution · Vulnerability · Healthcare Access Gap · Philippines & Taiwan · NASA satellite PM2.5 2022</sup>',
        x=0.5,
        xanchor='center',
        font=dict(size=16)
    ),
    hoverlabel=dict(
        bgcolor='white',
        bordercolor='#1B5E8A',
        font=dict(size=13, family='Arial'),
        align='left'
    )
)

# build bar chart
fig_bar = go.Figure(go.Bar(
    x=df_sorted_asia['overlap_score'],
    y=df_sorted_asia['display_name'],
    orientation='h',
    marker=dict(
        color=df_sorted_asia['overlap_score'],
        colorscale='RdYlGn_r',
        showscale=False
    ),
    text=df_sorted_asia['overlap_score'],
    textposition='outside',
    customdata=df_sorted_asia[['pm25','elderly','children_u5','hospital_access_count','country']].values,
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Country: %{customdata[4]}<br>'
        'Risk overlap score: %{x}<br>'
        'PM2.5: %{customdata[0]} ug/m3<br>'
        'Elderly (60+): %{customdata[1]:,}<br>'
        'Children under 5: %{customdata[2]:,}<br>'
        'Healthcare access: %{customdata[3]:,}'
        '<extra></extra>'
    )
))

fig_bar.update_layout(
    height=2800,
    margin=dict(l=180, r=100, t=80, b=40),
    title=dict(
        text='Risk Rankings — Philippines & Taiwan (110 units)<br><sup>Higher score = greater compounded health risk</sup>',
        x=0.5,
        xanchor='center',
        font=dict(size=16)
    ),
    xaxis=dict(title='Risk overlap score', range=[0, 0.85]),
    yaxis=dict(tickfont=dict(size=11))
)

fig_map.show()
print("figures built successfully")

In [ ]:
# combined t&p dashboard
map_html = fig_map.to_html(full_html=False, include_plotlyjs=True, config={'scrollZoom': True, 'displayModeBar': False})
bar_html = fig_bar.to_html(full_html=False, include_plotlyjs=False, config={'displayModeBar': False})
province_json = df_sorted_asia[['display_name', 'overlap_score', 'country']].to_json(orient='records')

dashboard = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Asia Climate-Health Risk Index</title>
    <style>
        * {{ box-sizing: border-box; margin: 0; padding: 0; font-family: Arial, sans-serif; }}
        body {{ background: #f4f4f4; }}
        .header {{ background: #1B5E8A; color: white; padding: 20px 30px; }}
        .header h1 {{ font-size: 20px; font-weight: 600; margin-bottom: 6px; }}
        .header p {{ font-size: 13px; opacity: 0.85; }}
        .tabs {{ display: flex; background: #ffffff; border-bottom: 2px solid #1B5E8A; padding: 0 30px; }}
        .tab {{ padding: 12px 24px; cursor: pointer; font-size: 14px; color: #555; border-bottom: 3px solid transparent; margin-bottom: -2px; transition: all 0.2s; }}
        .tab.active {{ background: #1B5E8A; color: white; font-weight: 600; border-bottom: 3px solid #1B5E8A; }}
        .panel {{ display: none; padding: 20px 30px; }}
        .panel.active {{ display: block; }}
        .map-container {{ background: white; border-radius: 8px; overflow: hidden; box-shadow: 0 1px 4px rgba(0,0,0,0.1); position: relative; }}
        .zoom-buttons {{ position: absolute; bottom: 30px; left: 20px; z-index: 999; display: flex; flex-direction: column; gap: 4px; }}
        .zoom-btn {{ width: 32px; height: 32px; background: white; border: 1px solid #ccc; border-radius: 4px; font-size: 18px; cursor: pointer; display: flex; align-items: center; justify-content: center; box-shadow: 0 1px 3px rgba(0,0,0,0.2); color: #333; }}
        .zoom-btn:hover {{ background: #f0f0f0; }}
        .bar-container {{ background: white; border-radius: 8px; overflow: hidden; box-shadow: 0 1px 4px rgba(0,0,0,0.1); }}
        .search-wrap {{ padding: 16px 20px 8px; background: white; border-radius: 8px 8px 0 0; display: flex; align-items: center; gap: 12px; border-bottom: 0.5px solid #eee; flex-wrap: wrap; }}
        .search-input {{ flex: 1; padding: 8px 14px; border: 1px solid #ccc; border-radius: 6px; font-size: 14px; outline: none; max-width: 360px; }}
        .search-input:focus {{ border-color: #1B5E8A; }}
        .search-label {{ font-size: 13px; color: #666; }}
        .match-count {{ font-size: 13px; color: #1B5E8A; font-weight: 600; min-width: 120px; }}
        .clear-btn {{ padding: 6px 12px; background: #f0f0f0; border: 1px solid #ccc; border-radius: 6px; font-size: 13px; cursor: pointer; color: #555; }}
        .clear-btn:hover {{ background: #e0e0e0; }}
        .country-filter {{ padding: 6px 12px; border: 1px solid #ccc; border-radius: 6px; font-size: 13px; color: #444; background: white; cursor: pointer; }}
        .meta {{ font-size: 12px; color: #888; margin-top: 10px; padding: 0 4px; }}
        .methodology {{ background: white; border-radius: 8px; padding: 16px 20px; margin-top: 16px; box-shadow: 0 1px 4px rgba(0,0,0,0.1); font-size: 13px; color: #444; line-height: 1.7; }}
        .methodology h3 {{ font-size: 14px; color: #1B5E8A; margin-bottom: 10px; font-weight: 600; }}
        .indicator-row {{ display: flex; gap: 12px; flex-wrap: wrap; margin-top: 10px; }}
        .indicator {{ background: #f0f5fa; border-radius: 6px; padding: 10px 14px; flex: 1; min-width: 180px; }}
        .indicator-name {{ font-weight: 600; color: #1B5E8A; font-size: 13px; margin-bottom: 4px; }}
        .indicator-desc {{ font-size: 12px; color: #666; }}
        .indicator-weight {{ font-size: 12px; font-weight: 600; color: #444; margin-top: 4px; }}
        .footer {{ background: #1B5E8A; color: rgba(255,255,255,0.7); font-size: 12px; padding: 12px 30px; margin-top: 30px; }}
    </style>
</head>
<body>
    <div class="header">
        <h1>Asia Climate-Health Risk Index</h1>
        <p>Air Pollution · Population Vulnerability · Healthcare Access Gap · Philippines & Taiwan · 110 units · NASA Satellite PM2.5 2022</p>
    </div>
    <div class="tabs">
        <div class="tab active" onclick="showTab('map', this)">Risk Map</div>
        <div class="tab" onclick="showTab('bar', this)">Rankings</div>
        <div class="tab" onclick="showTab('about', this)">About this index</div>
    </div>
    <div id="tab-map" class="panel active">
        <div class="map-container">
            {map_html}
            <div class="zoom-buttons">
                <button class="zoom-btn" onclick="zoomIn()">+</button>
                <button class="zoom-btn" onclick="zoomOut()">−</button>
            </div>
        </div>
        <p class="meta">Hover over any province or county to see detailed indicators. Scroll to zoom, click and drag to pan. Red = highest risk, Green = lowest.</p>
    </div>
    <div id="tab-bar" class="panel">
        <div class="search-wrap">
            <span class="search-label">Filter by country:</span>
            <select class="country-filter" id="country-filter" onchange="filterCountry(this.value)">
                <option value="all">All countries</option>
                <option value="Philippines">Philippines</option>
                <option value="Taiwan">Taiwan</option>
            </select>
            <span class="search-label">Search:</span>
            <input class="search-input" id="province-search" placeholder="Type a province or county name..." oninput="highlightProvince(this.value)">
            <span class="match-count" id="match-count"></span>
            <button class="clear-btn" onclick="clearSearch()">Clear</button>
        </div>
        <div class="bar-container" id="bar-chart-container">
            {bar_html}
        </div>
        <p class="meta">110 units ranked by health risk overlap score. Filter by country or search by name. Hover for full indicator breakdown.</p>
    </div>
    <div id="tab-about" class="panel">
        <div class="methodology">
            <h3>How the health risk overlap score is calculated</h3>
            <p>The overlap score is a composite index ranging from 0 to 1 combining three dimensions of health risk. A higher score means greater compounded risk across all three dimensions simultaneously.</p>
            <div class="indicator-row">
                <div class="indicator">
                    <div class="indicator-name">Air pollution — PM2.5</div>
                    <div class="indicator-desc">Annual mean fine particulate matter (μg/m³) from NASA satellite data (MODIS, MISR, VIIRS, 2022). Strongest direct evidence linking air pollution to cardiovascular mortality.</div>
                    <div class="indicator-weight">Weight: 40%</div>
                </div>
                <div class="indicator">
                    <div class="indicator-name">Population vulnerability</div>
                    <div class="indicator-desc">Elderly population (60+) and children under 5 — most vulnerable to air pollution and least able to cope with health shocks.</div>
                    <div class="indicator-weight">Elderly: 20% · Children: 10%</div>
                </div>
                <div class="indicator">
                    <div class="indicator-name">Healthcare access gap</div>
                    <div class="indicator-desc">Philippines: estimated population within 30min of hospital (HDX travel-time model). Taiwan: hospitals per 10,000 population (MOHW 2024). Both inverted so lower access = higher risk.</div>
                    <div class="indicator-weight">Weight: 30%</div>
                </div>
            </div>
            <p style="margin-top:14px"><b>Formula:</b> Overlap score = (PM2.5 × 0.40) + (Elderly × 0.20) + (Children × 0.10) + (Access gap × 0.30)</p>
            <p style="margin-top:8px"><b>Data sources:</b> NASA SEDAC V5.GL.04 (2022) · HDX PHL Risk Assessment · PSA · OCHA · Taiwan MOHW 2024 · GADM boundaries</p>
            <p style="margin-top:8px;color:#cc6600"><b>Cross-country note:</b> The healthcare access indicator uses different methodologies for Philippines and Taiwan and is not directly comparable between countries. See methodology documentation for details.</p>
            <p style="margin-top:8px"><b>Limitations:</b> PM2.5 reflects satellite estimates not ground measurements. AI-assisted Python development.</p>
        </div>
    </div>
    <div class="footer">
        PM2.5: NASA SEDAC 2022 · Philippines vulnerability: PSA 2020 · Philippines access: OCHA · Taiwan demographics: estimated · Taiwan hospitals: MOHW 2024 · Built with Python, Plotly and Mapbox
    </div>
    <script>
        function showTab(name, el) {{
            document.querySelectorAll('.panel').forEach(p => p.classList.remove('active'));
            document.querySelectorAll('.tab').forEach(t => t.classList.remove('active'));
            document.getElementById('tab-' + name).classList.add('active');
            el.classList.add('active');
        }}
        var currentZoom = 4;
        function zoomIn() {{
            currentZoom = Math.min(currentZoom + 0.5, 12);
            var mapDiv = document.querySelector('.js-plotly-plot');
            if (mapDiv) Plotly.relayout(mapDiv, {{'mapbox.zoom': currentZoom}});
        }}
        function zoomOut() {{
            currentZoom = Math.max(currentZoom - 0.5, 1);
            var mapDiv = document.querySelector('.js-plotly-plot');
            if (mapDiv) Plotly.relayout(mapDiv, {{'mapbox.zoom': currentZoom}});
        }}
        var provinceData = {province_json};
        function filterCountry(country) {{
            var barDiv = document.getElementById('bar-chart-container').querySelector('.js-plotly-plot');
            if (!barDiv) return;
            if (country === 'all') {{
                Plotly.restyle(barDiv, {{'marker.opacity': [provinceData.map(function() {{ return 1; }})]}});
                document.getElementById('match-count').textContent = '';
                return;
            }}
            var opacities = provinceData.map(function(d) {{ return d.country === country ? 1 : 0.15; }});
            Plotly.restyle(barDiv, {{'marker.opacity': [opacities]}});
            var count = provinceData.filter(function(d) {{ return d.country === country; }}).length;
            document.getElementById('match-count').textContent = count + ' units';
            document.getElementById('match-count').style.color = '#1B5E8A';
        }}
        function highlightProvince(query) {{
            var barDiv = document.getElementById('bar-chart-container').querySelector('.js-plotly-plot');
            if (!barDiv) return;
            var matchCount = document.getElementById('match-count');
            if (!query || query.length < 2) {{
                Plotly.restyle(barDiv, {{'marker.opacity': [provinceData.map(function() {{ return 1; }})], 'marker.line.width': [provinceData.map(function() {{ return 0; }})]}});
                matchCount.textContent = '';
                return;
            }}
            var q = query.toLowerCase();
            var matches = 0;
            var opacities = provinceData.map(function(d) {{ var m = d.display_name.toLowerCase().indexOf(q) !== -1; if (m) matches++; return m ? 1 : 0.15; }});
            var lineWidths = provinceData.map(function(d) {{ return d.display_name.toLowerCase().indexOf(q) !== -1 ? 2 : 0; }});
            Plotly.restyle(barDiv, {{'marker.opacity': [opacities], 'marker.line.width': [lineWidths], 'marker.line.color': ['#1B5E8A']}});
            matchCount.textContent = matches === 0 ? 'No match found' : matches + ' matched';
            matchCount.style.color = matches === 0 ? '#cc0000' : '#1B5E8A';
        }}
        function clearSearch() {{
            document.getElementById('province-search').value = '';
            document.getElementById('country-filter').value = 'all';
            highlightProvince('');
        }}
        var mapDiv2;
        function initMapHover() {{
            mapDiv2 = document.querySelector('#tab-map .js-plotly-plot');
            if (!mapDiv2) {{ setTimeout(initMapHover, 500); return; }}
            mapDiv2.on('plotly_hover', function(data) {{
                var idx = data.points[0].pointIndex;
                var n = {len(asia_master)};
                var lw = Array(n).fill(0.5);
                var lc = Array(n).fill('white');
                lw[idx] = 3; lc[idx] = '#FFD700';
                Plotly.restyle(mapDiv2, {{'marker.line.width': [lw], 'marker.line.color': [lc]}});
            }});
            mapDiv2.on('plotly_unhover', function() {{
                var n = {len(asia_master)};
                Plotly.restyle(mapDiv2, {{'marker.line.width': [Array(n).fill(0.5)], 'marker.line.color': [Array(n).fill('white')]}});
            }});
        }}
        function applyHoverLabel() {{
            var mapDiv2 = document.querySelector('#tab-map .js-plotly-plot');
            if (!mapDiv2) {{ setTimeout(applyHoverLabel, 500); return; }}
            Plotly.relayout(mapDiv2, {{
                'hoverlabel.bgcolor': 'white',
                'hoverlabel.bordercolor': '#1B5E8A',
                'hoverlabel.font.size': 13,
                'hoverlabel.font.family': 'Arial',
                'hoverlabel.align': 'left'
            }});
        }}
        initMapHover();
        applyHoverLabel();
    </script>
</body>
</html>"""

with open(BASE + 'asia_health_risk_dashboard_final.html', 'w') as f:
    f.write(dashboard)

print("Asia dashboard saved successfully")

In [ ]:
master_risk_final = pd.read_csv(BASE + 'master_risk_final_v2.csv')
asia_master = pd.read_csv(BASE + 'asia_master_v1.csv')

with open(BASE + 'asia_risk_v1.geojson') as f:
    geojson_asia = json.load(f)

master_risk_final['rank'] = range(1, len(master_risk_final) + 1)
master_risk_final['country'] = 'Philippines'

name_map = {
    'Metropolitan Manila First District': 'Manila City (1st District)',
    'Metropolitan Manila Second District': 'Quezon City (2nd District)',
    'Metropolitan Manila Third District': 'Caloocan & Others (3rd District)',
    'Metropolitan Manila Fourth District': 'Pasig & Others (4th District)',
    'Cotabato (North Cotabato)': 'North Cotabato',
    'Samar (Western Samar)': 'Western Samar',
    'Davao de Oro (Compostela Valley)': 'Davao de Oro',
    'City of Isabela (not a province)': 'Isabela City',
    'Special Geographic Area': 'BARMM Spec. Area'
}
master_risk_final['display_name'] = master_risk_final['ADM2_EN'].replace(name_map)
df_sorted_asia = asia_master.sort_values('overlap_score', ascending=True)

print("data loaded")
print("Philippines:", len(master_risk_final), "provinces")
print("Asia master:", len(asia_master), "units")

In [ ]:
mapbox_token = "pk.eyJ1IjoieW5uZ3V5ZW4iLCJhIjoiY21teDR3NzN2MTE0czJ1b2pjcTJ6d3Y2aCJ9.ATgZczFgyaW9dcHWCBMdzg"

In [ ]:
df_sorted_asia = asia_master.sort_values('overlap_score', ascending=True)

fig_map = go.Figure(go.Choroplethmapbox(
    geojson=geojson_asia,
    locations=asia_master['unit_id'],
    z=asia_master['overlap_score'],
    featureidkey='properties.unit_id',
    colorscale='RdYlGn_r',
    zmin=asia_master['overlap_score'].min(),
    zmax=asia_master['overlap_score'].max(),
    marker_opacity=0.7,
    marker_line_width=0.5,
    marker_line_color='white',
    text=asia_master['display_name'],
    customdata=asia_master[['pm25','elderly','children_u5','rural_pop_perc','hospital_access_count','rank','country']].values,
    hovertemplate=(
        '<span style="font-size:15px"><b>%{text}</b></span><br>'
        '<span style="color:#888">%{customdata[6]} · Rank %{customdata[5]:.0f}</span><br>'
        '<br>'
        '<b>Risk overlap score: %{z}</b><br>'
        '<br>'
        'PM2.5 (satellite 2022): %{customdata[0]} ug/m3<br>'
        'Elderly population (60+): %{customdata[1]:,}<br>'
        'Children under 5: %{customdata[2]:,}<br>'
        'Rural population: %{customdata[3]}%<br>'
        'Healthcare access: %{customdata[4]:,}'
        '<extra></extra>'
    ),
    colorbar=dict(
        title=dict(
            text='Health risk<br>overlap score',
            font=dict(size=12),
            side='top'
        ),
        thickness=20,
        len=0.75,
        x=0.98,
        tickfont=dict(size=12),
        tickvals=[
            asia_master['overlap_score'].min(),
            asia_master['overlap_score'].quantile(0.25),
            asia_master['overlap_score'].median(),
            asia_master['overlap_score'].quantile(0.75),
            asia_master['overlap_score'].max()
        ],
        ticktext=[
            str(round(asia_master['overlap_score'].min(), 2)),
            str(round(asia_master['overlap_score'].quantile(0.25), 2)),
            str(round(asia_master['overlap_score'].median(), 2)),
            str(round(asia_master['overlap_score'].quantile(0.75), 2)),
            str(round(asia_master['overlap_score'].max(), 2))
        ],
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(0,0,0,0.2)',
        borderwidth=1
    )
))

fig_map.update_layout(
    mapbox=dict(
        accesstoken=mapbox_token,
        style='mapbox://styles/mapbox/satellite-streets-v12',
        center=dict(lat=18, lon=118),
        zoom=4
    ),
    margin=dict(l=0, r=0, t=80, b=0),
    height=650,
    title=dict(
        text='Asia Climate-Health Risk Index<br><sup>Air Pollution · Vulnerability · Healthcare Access Gap · Philippines & Taiwan · NASA satellite PM2.5 2022</sup>',
        x=0.5,
        xanchor='center',
        font=dict(size=16)
    ),
    hoverlabel=dict(
        bgcolor='white',
        bordercolor='#1B5E8A',
        font=dict(size=13, family='Arial'),
        align='left'
    )
)

fig_bar = go.Figure(go.Bar(
    x=df_sorted_asia['overlap_score'],
    y=df_sorted_asia['display_name'],
    orientation='h',
    marker=dict(
        color=df_sorted_asia['overlap_score'],
        colorscale='RdYlGn_r',
        showscale=False
    ),
    text=df_sorted_asia['overlap_score'],
    textposition='outside',
    customdata=df_sorted_asia[['pm25','elderly','children_u5','hospital_access_count','country']].values,
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Country: %{customdata[4]}<br>'
        'Risk overlap score: %{x}<br>'
        'PM2.5: %{customdata[0]} ug/m3<br>'
        'Elderly (60+): %{customdata[1]:,}<br>'
        'Children under 5: %{customdata[2]:,}<br>'
        'Healthcare access: %{customdata[3]:,}'
        '<extra></extra>'
    )
))

fig_bar.update_layout(
    height=2800,
    margin=dict(l=180, r=100, t=80, b=40),
    title=dict(
        text='Risk Rankings — Philippines & Taiwan (110 units)<br><sup>Higher score = greater compounded health risk</sup>',
        x=0.5,
        xanchor='center',
        font=dict(size=16)
    ),
    xaxis=dict(title='Risk overlap score', range=[0, 0.85]),
    yaxis=dict(tickfont=dict(size=11))
)

print("figures built successfully")

In [ ]:
# startup cells
from google.colab import drive
drive.mount('/drive', force_remount=True)

import pandas as pd
import geopandas as gpd
import json
import plotly.graph_objects as go
from rasterstats import zonal_stats

BASE = '/drive/MyDrive/asia-health-risk-geodatabase/'
print("libraries loaded")

In [ ]:
master_risk_final = pd.read_csv(BASE + 'master_risk_final_v2.csv')
asia_master = pd.read_csv(BASE + 'asia_master_v1.csv')

with open(BASE + 'asia_risk_v1.geojson') as f:
    geojson_asia = json.load(f)

df_sorted_asia = asia_master.sort_values('overlap_score', ascending=True)
print("data loaded — asia units:", len(asia_master))

In [ ]:
mapbox_token = "pk.eyJ1IjoieW5uZ3V5ZW4iLCJhIjoiY21teDR3NzN2MTE0czJ1b2pjcTJ6d3Y2aCJ9.ATgZczFgyaW9dcHWCBMdzg"
print("token set")

In [ ]:
# add feature id to geojson
for feature in geojson_asia['features']:
    feature['id'] = feature['properties']['unit_id']

print("first feature id:", geojson_asia['features'][0]['id'])
print("total features:", len(geojson_asia['features']))

In [ ]:
# check first feature in geojson
first_feature = geojson_asia['features'][0]
print("feature ID key:", first_feature.get('id'))
print("properties:", list(first_feature['properties'].keys())[:5])
print("unit_id value", first_feature['properties'].get('unit_id'))

In [ ]:
import os
size = os.path.getsize(BASE + 'asia_health_risk_dashboard_final.html')
print(f"file size: {size / (1024*1024):.1f} MB")

In [ ]:
# fix geojson feature ids and save
for feature in geojson_asia['features']:
    feature['id'] = feature['properties']['unit_id']

import json
with open(BASE + 'asia_risk_v1.geojson', 'w') as f:
    json.dump(geojson_asia, f)

print("GeoJSON fixed and saved")
print("sample IDs:", [f['id'] for f in geojson_asia['features'][:3]])
print("Taiwan sample:", [f['id'] for f in geojson_asia['features'] if 'TWN' in str(f['id'])][:3])

In [ ]:
#integrating indonesia
indonesia = gpd.read_file(BASE + 'gadm41_IDN_1.json')
print("Indoensia shape:", indonesia.shape)
print("columns:", indonesia.columns.tolist())
print(indonesia[['NAME_1', 'GID_1']].to_string())

In [ ]:
tif_path = BASE + 'sdei-global-annual-gwr-pm2-5-modis-misr-seawifs-viirs-aod-v5-gl-04-2022-geotiff.tif'
nodata_value = -3.3999999521443642e+38

stats = zonal_stats(
    indonesia,
    tif_path,
    stats=['mean'],
    nodata=nodata_value
)

indonesia['pm25'] = [s['mean'] for s in stats]
indonesia['pm25'] = indonesia['pm25'].round(2)
print(indonesia[['NAME_1', 'pm25']].sort_values('pm25', ascending=False).to_string())

In [ ]:
# load indonesia files
idn_geo = gpd.read_file(BASE + 'gadm41_IDN_2.json')
idn_vuln = pd.read_csv(BASE + 'IDN_ADM2_vulnerability.csv')
idn_demo = pd.read_csv(BASE + 'IDN_ADM2_demographics.csv')
idn_acc = pd.read_csv(BASE + 'IDN_ADM2_access.csv')

print("IDN boundaries:", idn_geo.shape)
print("IDN vulnerability:", idn_vuln.shape)
print("IDN demographics:", idn_demo.shape)
print("IDN access:", idn_acc.shape)

print("\nBoundary columns:", idn_geo.columns.tolist()[:6])
print("Vulnerability columns:", idn_vuln.columns.tolist())
print("Access columns:", idn_acc.columns.tolist())

In [ ]:
import os
files = os.listdir(BASE)
idn_files = [f for f in files if 'IDN' in f or 'idn' in f.lower() or 'indonesia' in f.lower()]
print("Indonesia files found:", idn_files)
print("\nAll files:", sorted(files))

In [ ]:
print("Boundary ID column:", idn_geo.columns.tolist())
print("Sample boundary IDs:", idn_geo['GID_2'].head(5).tolist())
print("Sample HDX IDs:", idn_vuln['ADM2_PCODE'].head(5).tolist())